# STT Evaluation Pipeline (발표 STT 평가 파이프라인)

발표 연습 한 번(take)의 **STT(음성 인식 결과)** 를, 대본 분석 노트북(`script_analysis_pipeline.ipynb`)이 만들어 DB 에 저장한
**Evaluation Rubric(평가 기준)** 으로 채점합니다. 대본은 다시 분석하지 않고 저장된 평가 기준을 읽어 씁니다.
아래 그림의 괄호는 이 노트북의 장 번호입니다.

```text
   발표 연습 STT (슬라이드별)                 Evaluation Rubric (대본 분석 노트북 → DB)
             │                                          │
      STT 정규화 (2장)                                  │
             ├─────────────────────┬────────────────────┘
             ▼                     ▼
   규칙 분석 (3·4장)            LLM API 의미 평가 1회 (5장)
   · 문장 정렬 · 대본 충실도     · 대본 문장마다 근거 → 이유 → 판정
   · 핵심 사실 검증 (수치·이름)  · 규칙이 못 찾은 영문 이름 확인
   · 비슷한 말 찾기 (4-3)
     발음이 비슷한 단어·수치 → 판단 보류: 비율에서 빼고 개수·위치만 남김
             └──────────┬──────────┘
                        ▼
          문장별 병합 · 충돌 검사 (6장, 코드)
                        ▼  충돌한 문장이 있는 슬라이드만
          LLM API 교차 검증 1회 (7장) — 충돌한 문장 재판정
                        ▼
          Key Point 판정 = 문장 판정 모으기 (6장, 코드)
                        ▼
          점수 계산 (8장, 코드) → DB (9장)
          · 평가 결과 · 비슷한 말(위치 포함) → 나중에 코칭 agent 가 판단
```

**구조** — 슬라이드마다 규칙 분석 1회와 LLM 의미 평가 1회, 규칙과 LLM 이 어긋난 슬라이드만 교차 검증 1회를 더 부릅니다.
LLM 은 판정만 하고 점수는 코드가 계산합니다.

**전제** — 웹이 슬라이드 넘김을 기록하므로 슬라이드별 STT 는 그 슬라이드를 띄워 둔 동안 말한 내용입니다.
음성 인식은 이어폰 마이크로 녹음한 Deepgram 결과라 대체로 정확하지만, 숫자는 가끔 잘못 받아 적을 수 있습니다.

### 설계 포인트

| 어려운 점 | 단순한 방법이면 | 이 노트북의 방법 |
|---|---|---|
| STT 는 숫자를 들리는 대로 적는다 (`사십이 퍼센트`, `만 구천 원`) | 문자열 비교로는 `42%` 를 못 찾아 누락으로 오판 | 대본과 **같은 수 파서**로 값을 읽어 정규형(`42%`)끼리 비교 (4장) |
| 발표자는 문장을 합치고 쪼개고 순서를 바꾼다 | 문장 1:1 비교면 의역을 누락으로 오판 | 대본 문장마다 STT **연속 1~3문장 구간**과 정렬 (3장) |
| 영문 이름을 한글로 적는다 (`SeatFlow` → `시트플로우`) | 규칙은 못 찾아 누락으로 오판 | 규칙이 못 찾은 영문 이름만 **LLM 이 같은 호출에서 확인** (4·5장) |
| 음성 인식이 발음이 비슷한 다른 말로 적는다 (`노쇼` → `노조`, `삼십 분` → `사십 분`) | 발표자가 틀린 것으로 감점 | 텍스트만으로는 발표자 실수인지 인식 오류인지 가릴 수 없으므로 **판단을 보류**: 발음이 비슷한 단어·수치를 규칙으로 찾아 **비율 점수에서 빼고 개수만** 점수에 넣고, **위치와 함께 저장**해 나중에 코칭 agent 가 판단 (4-3·8·9장) |
| LLM 판정은 가끔 틀리고 흔들린다 | 한 번의 판정이 그대로 점수가 됨 | 규칙과 LLM 을 **독립적으로** 돌리고, 어긋난 Key Point 만 근거를 모아 **다시 판정** (6·7장) |
| 여러 문장으로 된 Key Point 를 통째로 판정하면 경계(일부 / 빠짐)가 흔들린다 | 같은 발화도 채점할 때마다 판정이 바뀜 | LLM 은 **대본 문장 하나씩** 판정하고, Key Point 판정은 **코드가 문장 판정을 모아** 정함 (5·6장) |
| LLM 이 근거 없이 판정할 수 있다 | 판정을 확인할 방법이 없음 | **근거 문장 번호 → 이유 → 판정** 순서로 받고, 근거 번호를 코드가 확인 (5·6장) |
| 점수 스케일이 흔들린다 | LLM 이 점수를 매기면 같은 판정도 점수가 달라짐 | LLM 은 판정만, **점수는 코드** (8장) |
| 잘 되는지 알 수 없다 | — | 가상 STT 에 **정답 라벨**을 두고 규칙만 / LLM 1차 / 최종을 비교 (11장) |
| LLM 판정이 채점할 때마다 다를 수 있다 | — | 같은 STT 를 여러 번 채점해 판정·점수가 얼마나 달라지는지 측정 (12장) |

## 0. 대본 분석 노트북 불러오기

대본 분석 노트북을 `%run` 으로 실행해 그 함수(대본 정규화, 수 파서, Kiwi, DB 연결, `SCORE_WEIGHT` 등)와 평가 기준을 그대로 씁니다.
- 대본 분석의 LLM 응답은 DB 캐시에 있으므로, 대본이 바뀌지 않았다면 API 를 다시 부르지 않습니다.
- 대본 분석 노트북의 일관성 측정(추가 호출)은 `RUBRIC_CONSISTENCY_SAMPLES=1` 로 끕니다.
- 대본 분석 노트북의 출력은 `script_analysis_log` 에 담아 숨깁니다 (`script_analysis_log.show()` 로 볼 수 있습니다).

In [1]:
%%capture script_analysis_log
# 대본 분석 노트북을 실행해 함수와 평가 기준(DB)을 그대로 가져온다.
# LLM 응답은 캐시를 쓰므로 대본이 바뀌지 않았다면 API 를 부르지 않는다. 일관성 측정(추가 호출)은 끈다.
import os
os.environ["RUBRIC_CONSISTENCY_SAMPLES"] = "1"
%run ./script_analysis_pipeline.ipynb

In [2]:
from collections import defaultdict
from difflib import SequenceMatcher
from typing import NamedTuple

STT_DIR = ROOT / "data" / "stt"
LABEL_DIR = ROOT / "data" / "stt_labels"  # 정답 라벨. 13장 성능 측정에서만 읽고, 평가 파이프라인은 보지 않는다

print(f"MODEL={MODEL}")
print("평가 기준을 불러온 대본:", {name: len(rubrics) for name, rubrics in all_rubrics.items()})

MODEL=openai/gpt-5.6-luna
평가 기준을 불러온 대본: {'가상대본1': 9, '가상대본2': 11}


## 1. 가상 STT 데이터

`data/stt/` 에 대본(`data/scripts/`)마다 연습 9번의 STT 가 슬라이드별로 있습니다. 대본을 바탕으로 만든 가상 데이터이고,
실제 음성 인식 결과처럼 문장부호가 거의 없고 간투사(`음`, `어`)·말 반복·띄어쓰기 오류가 섞여 있습니다.
숫자의 일부는 한글로(`사십이 퍼센트`, `천이백사십만 건`, `오후 여섯 시`), 영문 이름의 일부는 발음대로(`시트플로우`, `엑스지부스트`) 적혀 있습니다.

| take | 시나리오 | 내용 |
|---|---|---|
| take1 | 충실 | 대본을 거의 그대로 말함 |
| take2 | 의역 | 뜻은 같지만 자기 말로 바꾸고, 문장을 합치거나 순서를 바꿈. 수치·이름은 맞게 말함 |
| take3 | 누락 | 문장의 25~40% 를 빠뜨리고, 일부 수치를 `많이`, `꽤` 로 뭉뚱그림 |
| take4 | 실수 | 대부분 말했지만 틀린 수치, 대본과 반대되는 말, 즉흥 발언, 틀렸다가 바로 고쳐 말한 경우가 섞임 |
| take5 | 혼합 | 의역·누락·수치 뭉뚱그림·틀린 수치·반대되는 말·즉흥 발언이 한 연습에 섞임 (실제 연습에 가까운 경우) |
| take6 | 인식오류 | 대본대로 말했지만 음성 인식이 수치 4~6개를 잘못 적음 (`삼백십이` → `사백십이`, `42%` → `40%`). 발표자가 실제로 틀린 수치 2개도 섞음 |
| take7 | 더듬기 | 간투사·말 끊김·고쳐 말하기가 많음. 이 연습은 음성 인식이 수치를 숫자로 적음 (`42%`, `1,240만 건`, `오후 6시`) |
| take8 | 요약 | 시간에 쫓겨 슬라이드마다 1~3문장으로 요약. 수치를 어림해 말함 (`40퍼센트 넘게`, `천만 건이 넘는`) |
| take9 | 발음 | 대본대로 말했지만 음성 인식이 발음이 비슷한 단어 7~8개와 수치 1~2개를 잘못 적음 (`좌석` → `자석`, `모델` → `모텔`). 발표자가 실제로 다른 단어를 쓴 곳 2개와 같은 뜻의 다른 말도 섞음 |

`data/stt_labels/` 에는 대본 문장마다 실제로 어떻게 말했는지 적은 **정답 라벨**이 있습니다
(verbatim / paraphrased / partial / missing / contradicted, 빠뜨린 값, 틀리게 말한 값, 음성 인식이 잘못 적은 값, 어림해 말한 값).
라벨은 발표자가 **실제로 말한 것** 기준입니다. 음성 인식이 수치를 잘못 적었어도 발표자가 맞게 말했으면 전달입니다.
평가 파이프라인은 라벨을 보지 않고, 11장 성능 측정에서만 읽습니다.

In [3]:
class SlideSTT(BaseModel):
    slide_number: int
    stt: str = Field(description="이 슬라이드에서 발표자가 말한 내용의 음성 인식 결과")


class Take(BaseModel):
    """발표 연습 한 번의 STT. 슬라이드별로 나뉘어 있다."""
    script_name: str = Field(description="어느 대본의 연습인지 (대본 JSON 파일 이름)")
    take_id: str
    scenario: str = Field(description="가상 데이터의 시나리오 (충실 / 의역 / 누락 / 실수)")
    slides: list[SlideSTT]


def load_take(path: Path) -> Take:
    return Take.model_validate_json(path.read_text(encoding="utf-8"))


TAKE_FILES = sorted(STT_DIR.glob("*.json"))
takes = [load_take(p) for p in TAKE_FILES]

pd.DataFrame([
    {"take_id": t.take_id, "script": t.script_name, "scenario": t.scenario, "slides": len(t.slides),
     "chars": sum(len(s.stt) for s in t.slides), "slide 1 STT": t.slides[0].stt[:50] + "…"}
    for t in takes
])

,take_id,script,scenario,slides,chars,slide 1 STT
0,가상대본1_take1,가상대본1,충실,9,2043,안녕하십니까 음 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려주는 서비스 시트플로우를 …
1,가상대본1_take2,가상대본1,의역,9,2227,안녕하세요 저희는 빈자리연구소라는 팀이고요 음 도서관이 얼마나 붐빌지 예측해서 빈자리를 먼…
2,가상대본1_take3,가상대본1,누락,9,1297,안녕하십니까 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려주는 서비스 시트플로우를 만든…
3,가상대본1_take4,가상대본1,실수,9,2176,안녕하십니까 음 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려 주는 서비스 시트플로우를…
4,가상대본1_take5,가상대본1,혼합,9,1982,안녕하십니까 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려 드리는 서비스 시트플로우를 …
5,가상대본1_take6,가상대본1,인식오류,9,1998,안녕하십니까 음 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려 주는 서비스 SeatFl…
6,가상대본1_take7,가상대본1,더듬기,9,2036,어 안녕하십니까 제가 좀 긴장이 돼서요 저 저희는 음 도서관 좌석 혼잡도를 예측해서 빈자리…
7,가상대본1_take8,가상대본1,요약,9,955,안녕하십니까 빈자리연구소입니다 음 저희가 대학생 삼백 명 넘게 물어봤더니 빈자리 찾는데만 …
8,가상대본1_take9,가상대본1,발음,9,1976,안녕하십니까 음 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려 주는 서비스 SeatFl…
9,가상대본2_take1,가상대본2,충실,11,2183,안녕하세요 음 저희 프로젝트는 남는 빵을 줄이는 동네 빵집 재고 예보입니다 보통 동네 빵집…


## 2. STT 정규화

대본과 같은 정규화(NFKC, 따옴표 통일, 공백 정리, Kiwi 문장 분리)에 두 가지를 더합니다.
- **간투사 제거**: 단독으로 쓰인 `음`·`어`·`으`·`엄`·`흠`·`아`·`에`. `그러니까`, `이제` 처럼 뜻이 있을 수 있는 말은 남깁니다.
- **바로 반복된 말 제거**: `그 그 결과` → `그 결과`.

한글로 적힌 숫자는 텍스트를 고치지 않습니다. 4장에서 대본과 같은 수 파서로 값을 읽습니다.
대본 분석 노트북의 수 파서는 `사십이 퍼센트`, `만 구천 원`, `십일 점 사 퍼센트`, `이천이십오 년 삼 월`, `오후 여섯 시 반`, `삼 대 일` 같은
한글 표기도 읽고, Kiwi 가 수사(NR)로 분석한 경우만 수로 인정합니다 (`이 분이 오셨다` 의 `이` 는 수가 아님).

STT 문장에는 `[T0]`, `[T1]` … 번호를 붙입니다. 이 번호가 LLM 판정의 근거 표시에 쓰입니다.

**왜 LLM 이 아니라 규칙으로 정규화하나** — 정규화는 모든 단계의 입력이라, 여기서 텍스트가 바뀌면 뒤의 모든 판정이 그 위에서 이뤄집니다.
- **말한 내용을 바꾸지 않는다**: LLM 은 정리하면서 문맥에 맞게 고쳐 쓰는 경향이 있습니다(`노조 비율` → `노쇼 비율`, 어색한 수치 보정, 문장 합치기).
  그러면 음성 인식 오류도, 발표자의 실제 실수도 이 단계에서 사라집니다. 이 평가는 '실제로 말한 것'과 대본을 비교하므로, 규칙으로 지워도 되는 것(간투사·말 반복)만 지웁니다.
- **위치가 보존된다**: 비슷한 말(4-3)은 원본 STT 위치와 함께 저장해 녹음 구간을 찾는 데 씁니다. 문장을 다시 쓰면 원본과의 위치 대응이 사라집니다.
- **결과가 흔들리지 않는다**: 정규화부터 수치 검증까지 규칙이라 같은 STT 는 늘 같은 결과입니다 (12장 반복 채점에서 수치 점수 차이 0).
- **해석은 5장이 한다**: 띄어쓰기 오류, 한글로 적힌 숫자·이름, 고쳐 말하기 같은 STT 특유의 표기는 의미 평가 LLM 이 원문을 보면서 해석합니다. 텍스트를 고치지 않고 이해만 합니다.

실제 음성 인식 결과에서 문장이 어색하게 나뉘면(3-2 점검), LLM 으로 다시 쓰기보다 음성 인식의 구두점·문단 옵션을 먼저 확인하고,
그래도 부족하면 LLM 에게 텍스트는 그대로 두고 '끊을 위치'만 받는 방식이 안전합니다.

In [4]:
# 뜻 없는 간투사와 바로 반복된 말은 지운다. "그러니까", "이제" 처럼 뜻이 있을 수 있는 말은 남긴다
FILLER = re.compile(r"(?<![가-힣])(?:음+|어+|으+|엄+|흠+|아+|에+)(?![가-힣])")
REPEATED_WORD = re.compile(r"(?<![가-힣])([가-힣]{1,4})(?:\s+\1)+(?![가-힣])")


def normalize_stt(text: str) -> NormalizedScript:
    """STT 정규화: 대본과 같은 정규화에 간투사·말 반복 제거를 더한다.

    숫자를 한글로 받아 적은 표기("사십이 퍼센트")는 바꾸지 않는다. 4장 사실 검증에서 대본 쪽과 같은 수 파서로 읽는다.
    """
    text = unicodedata.normalize("NFKC", text).translate(QUOTE_MAP)
    text = FILLER.sub(" ", text)
    text = REPEATED_WORD.sub(r"\1", text)
    return normalize_script(text)


sample_take = takes[0]
sample_norm = normalize_stt(sample_take.slides[0].stt)
print("원문:", sample_take.slides[0].stt[:200])
print("정규화:", sample_norm.text[:200])
for s in sample_norm.sentences:
    print(f"[T{s.index}] {s.text}")

원문: 안녕하십니까 음 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려주는 서비스 시트플로우를 만든 빈자리 연구소입니다. 시험 기간만 되면 열람실이 항상 사람으로 꽉 차죠 저희가 대학생 삼백십이 명에게 물어보니까 빈자리를 찾느라 하루 평균 이십삼 분을 쓴다고 답했습니다 어 자리가 없는 것만 문제가 아닙니다 가방만 두고 자리를 비우는 경우가 많아서 실제로는 비어있
정규화: 안녕하십니까 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려주는 서비스 시트플로우를 만든 빈자리 연구소입니다. 시험 기간만 되면 열람실이 항상 사람으로 꽉 차죠 저희가 대학생 삼백십이 명에게 물어보니까 빈자리를 찾느라 하루 평균 이십삼 분을 쓴다고 답했습니다 자리가 없는 것만 문제가 아닙니다 가방만 두고 자리를 비우는 경우가 많아서 실제로는 비어있는 좌석
[T0] 안녕하십니까
[T1] 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려주는 서비스 시트플로우를 만든 빈자리 연구소입니다.
[T2] 시험 기간만 되면 열람실이 항상 사람으로 꽉 차죠
[T3] 저희가 대학생 삼백십이 명에게 물어보니까 빈자리를 찾느라 하루 평균 이십삼 분을 쓴다고 답했습니다
[T4] 자리가 없는 것만 문제가 아닙니다
[T5] 가방만 두고 자리를 비우는 경우가 많아서 실제로는 비어있는 좌석도 찾기가 어렵습니다
[T6] 그래서 저희는 이제 좌석이 언제 비는지 미리 알려 주면 이 시간을 줄일 수 있다고 생각했습니다


## 3. 문장 정렬과 대본 충실도 (규칙)

**내용 형태소** (`content_tokens`) — Kiwi 로 형태소를 나누고 조사·어미·간투사를 버린 뒤 명사, 동사·형용사 어간, 수사, 영문, 부사만 남깁니다.
`했습니다` 와 `했어요` 의 차이가 사라지고, 수치는 표기와 상관없이 같은 토큰이 됩니다 (`42%` 와 `사십이 퍼센트` 모두 `<percentage:42%>`).

**문장 정렬** (`align_sentences`) — 대본 문장마다, STT 의 연속 1~3문장 구간 중 대본 문장의 내용 형태소가 가장 많이 나온 구간을 찾습니다.
발표자는 문장을 합치거나 쪼개 말하므로 1:1 이 아니라 구간으로 맞추고, 비율이 같으면 짧은 구간을 고릅니다.
이 비율(`coverage`)이 6장에서 LLM 판정과 비교할 규칙 쪽 근거가 됩니다. 인사·전환 문장(`skip`)은 뺍니다.

**대본 충실도** (`script_fidelity`) — 대본과 STT 의 내용 형태소를 순서를 지키며 맞춘 뒤(`SequenceMatcher`),
recall(대본이 얼마나 나왔나)과 precision(STT 중 대본에 있던 말의 비율)의 조화평균을 냅니다.
뜻이 같아도 말을 바꾸면 낮아지므로 '전달했는가'가 아니라 **'대본대로 말했는가'** 를 보는 점수입니다.

In [5]:
# 내용 형태소: 조사·어미·간투사를 빼고 뜻을 가진 말만 비교한다 (동사·형용사는 어간이라 '했습니다' / '했어요' 차이가 사라진다)
CONTENT_TAGS = {"NNG", "NNP", "NR", "SN", "SL", "SH", "VV", "VA", "XR", "MAG"}
NUMERIC_TYPES = ("percentage", "money", "date", "time", "duration", "quantity", "ratio", "number")


class Alignment(BaseModel):
    sentence_index: int = Field(description="대본 문장 번호")
    stt_ids: list[int] = Field(description="가장 비슷한 STT 문장 번호 (연속 1~3문장)")
    coverage: float = Field(description="대본 문장의 내용 형태소 중 STT 에 나온 비율 (0~1)")


class Token(NamedTuple):
    key: str        # 비교에 쓰는 값: 형태소 원형(소문자) 또는 수치 자리표시 '<percentage:42%>'
    sentence: int
    start: int      # 정규화 텍스트 기준 위치
    end: int
    tag: str


def positioned_tokens(norm: NormalizedScript) -> list[Token]:
    """내용 형태소를 위치·품사와 함께. 수치는 표기와 상관없이 같은 토큰이 되도록 정규형으로 바꾼다 ('42%' = '사십이 퍼센트')."""
    numbers = {}
    for fact in extract_critical_facts(norm):
        if fact.type in NUMERIC_TYPES:
            for start, end in fact.spans:
                numbers[start] = (end, f"<{fact.type}:{fact.normalized}>")
    tokens, skip_until = [], -1
    for tok in Morphemes(norm.text).tokens:
        if tok.start < skip_until:
            continue
        sentence = next((s.index for s in norm.sentences if s.start <= tok.start < s.end), 0)
        if tok.start in numbers:
            skip_until, placeholder = numbers[tok.start]
            tokens.append(Token(placeholder, sentence, tok.start, skip_until, "NUM"))
        elif (tok.tag == "XSN" and tok.form != "들" and tokens and tokens[-1].tag in ("NNG", "NNP")
              and tokens[-1].end == tok.start and tokens[-1].sentence == sentence):
            # 명사에 붙은 접미사는 한 단어로: '이용'+'률' → '이용률' ('이용료' 와 구별된다). 복수 '들' 은 뗀다
            prev = tokens.pop()
            tokens.append(Token(prev.key + tok.form.lower(), sentence, prev.start, tok.start + tok.len, prev.tag))
        elif tok.tag in CONTENT_TAGS:
            tokens.append(Token(tok.form.lower(), sentence, tok.start, tok.start + tok.len, tok.tag))
    return tokens


def content_tokens(norm: NormalizedScript) -> list[tuple[str, int]]:
    """(내용 형태소, 문장 번호) 목록. 조사·어미·간투사를 빼고 뜻을 가진 말만."""
    return [(t.key, t.sentence) for t in positioned_tokens(norm)]


def align_sentences(script_norm: NormalizedScript, script_tokens, stt_norm: NormalizedScript, stt_tokens,
                    roles: list[str]) -> list[Alignment]:
    """대본 문장마다 가장 비슷한 STT 구간(연속 1~3문장)을 찾는다. skip 문장은 뺀다.

    발표자는 문장을 합치거나 쪼개 말하므로 1:1 이 아니라 구간으로 맞춘다. 이 정렬이 LLM 판정과 비교할 규칙 쪽 근거가 된다.
    """
    by_script = {i: Counter(t for t, s in script_tokens if s == i) for i in range(len(script_norm.sentences))}
    by_stt = [Counter(t for t, s in stt_tokens if s == j) for j in range(len(stt_norm.sentences))]
    alignments = []
    for i, role in enumerate(roles):
        target = by_script[i]
        if role == "skip" or not target:
            continue
        candidates = []
        for start in range(len(by_stt)):
            window = Counter()
            for end in range(start, min(start + 3, len(by_stt))):
                window += by_stt[end]
                coverage = sum((target & window).values()) / sum(target.values())
                candidates.append((round(coverage, 6), -(end - start), list(range(start, end + 1))))
        # 비율이 같으면 짧은 구간: 뜻 없는 앞뒤 문장까지 끌어오지 않는다
        coverage, _, ids = max(candidates, key=lambda c: (c[0], c[1]), default=(0.0, 0, []))
        alignments.append(Alignment(sentence_index=i, stt_ids=ids if coverage > 0 else [], coverage=round(coverage, 3)))
    return alignments


def script_fidelity(script_tokens, stt_tokens, roles: list[str]) -> dict:
    """대본 충실도: 대본(skip 제외)의 내용 형태소가 같은 순서로 STT 에 나온 정도.

    recall = 대본 쪽이 얼마나 그대로 나왔나, precision = STT 중 대본에 있던 말의 비율, fidelity = 둘의 조화평균.
    """
    a = [t for t, s in script_tokens if roles[s] != "skip"]
    b = [t for t, _ in stt_tokens]
    if not a or not b:
        return {"recall": 0.0, "precision": 0.0, "fidelity": 0.0}
    matched = sum(block.size for block in SequenceMatcher(None, a, b, autojunk=False).get_matching_blocks())
    recall, precision = matched / len(a), matched / len(b)
    fidelity = 2 * recall * precision / (recall + precision) if matched else 0.0
    return {"recall": round(recall, 3), "precision": round(precision, 3), "fidelity": round(fidelity, 3)}


# 예시: 첫 번째 연습의 슬라이드 1
_rubric = load_rubric(conn, sample_take.script_name, 1)
_script_norm = normalize_script(_rubric.normalized_script)
_s_tokens, _t_tokens = content_tokens(_script_norm), content_tokens(sample_norm)
display(pd.DataFrame([a.model_dump() for a in align_sentences(_script_norm, _s_tokens, sample_norm, _t_tokens, _rubric.sentence_roles)]))
print("대본 충실도:", script_fidelity(_s_tokens, _t_tokens, _rubric.sentence_roles))

,sentence_index,stt_ids,coverage
0,1,[1],0.923
1,2,[2],0.667
2,3,"[1, 2, 3]",1.000
3,4,[4],1.000
4,5,[5],1.000
5,6,[6],1.000


대본 충실도: {'recall': 0.918, 'precision': 0.818, 'fidelity': 0.865}


### 3-2. STT 문장 분리 점검 (규칙, API 호출 없음)

실제 음성 인식 결과는 문장부호가 거의 없어서 Kiwi 가 문장을 나눕니다. 여러 문장이 한 문장으로 붙으면 3장 정렬 구간과 5장 근거(T번호)가 거칠어집니다.
실제 Deepgram 결과를 넣었을 때 이 표로 먼저 확인합니다.

| 열 | 읽는 법 |
|---|---|
| 평균 길이 · 최대 길이 · 120자 넘는 문장 | 긴 문장이 많으면 여러 문장이 붙은 것 — 음성 인식의 구두점·문단 옵션을 확인 |
| 정렬 구간 1 / 2 / 3문장, 3문장 구간 비율 | 대본 한 문장이 STT 여러 문장에 걸친 정도. 발표자가 쪼개 말하면 자연스럽게 늘지만, 문장 분리가 잘게 쪼개진 경우에도 늘어남 |
| 정렬 안 된 대본 문장 | STT 에서 대본 단어를 하나도 못 찾은 문장 (빠뜨린 문장이거나 정렬 실패) |
| 남은 간투사 후보 | 뜻이 있을 수도 있어 지우지 않은 말(`그`, `저`, `뭐`, `이제` …)의 빈도. 실제 데이터에서 간투사로 많이 쓰이면 2장 목록에 넣을지 검토 |

In [6]:
# 실제 STT 는 문장부호가 거의 없어 Kiwi 가 문장을 나눈다. 여러 문장이 한 문장으로 붙으면 3장 정렬과 T번호 근거가 거칠어진다
LONG_SENTENCE = 120   # 이보다 긴 STT 문장(글자 수)은 여러 문장이 붙었을 가능성이 크다
# 뜻이 있을 수도 있어서 지우지 않는 말. 실제 데이터에서 간투사로 많이 쓰이면 2장 FILLER 에 넣을지 검토한다
FILLER_CANDIDATES = ["그", "저", "뭐", "막", "좀", "이제", "그러니까", "약간"]


def segmentation_report(takes: list[Take]) -> pd.DataFrame:
    """연습마다 STT 문장 분리와 정렬 상태를 요약한다 (규칙만, API 호출 없음)."""
    filler_pattern = re.compile(r"(?<![가-힣])(" + "|".join(FILLER_CANDIDATES) + r")(?![가-힣])")
    rows = []
    for take in takes:
        lengths, windows, fillers = [], Counter(), Counter()
        for s in take.slides:
            rubric = load_rubric(conn, take.script_name, s.slide_number)
            script_norm, stt_norm = normalize_script(rubric.normalized_script), normalize_stt(s.stt)
            lengths += [len(x.text) for x in stt_norm.sentences]
            s_tokens, t_tokens = positioned_tokens(script_norm), positioned_tokens(stt_norm)
            for a in align_sentences(script_norm, [(t.key, t.sentence) for t in s_tokens], stt_norm,
                                     [(t.key, t.sentence) for t in t_tokens], rubric.sentence_roles):
                windows[len(a.stt_ids)] += 1
            fillers.update(filler_pattern.findall(stt_norm.text))
        aligned = sum(windows.values())
        rows.append({
            "take_id": take.take_id, "STT 문장 수": len(lengths),
            "평균 길이": round(sum(lengths) / max(len(lengths), 1), 1), "최대 길이": max(lengths, default=0),
            f"{LONG_SENTENCE}자 넘는 문장": sum(n > LONG_SENTENCE for n in lengths),
            "정렬 구간 1 / 2 / 3문장": f"{windows[1]} / {windows[2]} / {windows[3]}",
            "3문장 구간 비율": round(windows[3] / max(aligned, 1), 2),
            "정렬 안 된 대본 문장": windows[0],
            "남은 간투사 후보": ", ".join(f"{w} {n}" for w, n in fillers.most_common(3)) or "-",
        })
    return pd.DataFrame(rows)


segmentation_report(takes)

,take_id,STT 문장 수,평균 길이,최대 길이,120자 넘는 문장,정렬 구간 1 / 2 / 3문장,3문장 구간 비율,정렬 안 된 대본 문장,남은 간투사 후보
0,가상대본1_take1,46,43.2,85,0,37 / 3 / 1,0.02,0,"이제 1, 그러니까 1, 그 1"
1,가상대본1_take2,42,52.0,114,0,28 / 8 / 5,0.12,0,"그 3, 그러니까 2, 좀 1"
2,가상대본1_take3,32,39.3,71,0,32 / 3 / 2,0.05,4,"그 1, 그러니까 1"
3,가상대본1_take4,46,46.3,96,0,38 / 2 / 1,0.02,0,"그 3, 그러니까 1, 좀 1"
4,가상대본1_take5,41,47.4,111,0,29 / 9 / 2,0.05,1,"좀 2, 그 2, 그러니까 1"
5,가상대본1_take6,42,46.6,126,1,34 / 2 / 5,0.12,0,"그 2, 그러니까 1, 좀 1"
6,가상대본1_take7,45,42.8,87,0,37 / 2 / 2,0.05,0,"그 14, 저 4, 좀 1"
7,가상대본1_take8,21,44.5,90,0,32 / 5 / 2,0.05,2,그 2
8,가상대본1_take9,45,42.9,87,0,39 / 0 / 2,0.05,0,"그러니까 1, 그 1, 이제 1"
9,가상대본2_take1,52,40.7,73,0,42 / 1 / 1,0.02,0,"그 4, 이제 1"


## 4. 핵심 사실 검증 (규칙)

평가 기준의 Critical Fact(수치·날짜·이름)가 STT 에 나왔는지 봅니다.

| 결과 | 조건 |
|---|---|
| `matched` | 같은 값을 말함. 수치는 STT 에서도 같은 수 파서로 읽어 정규형끼리 비교 (`사십이 퍼센트` = `42%`). 이름은 공백·문장부호를 무시하고 찾음 |
| `mismatched` | 같은 값은 없고, 그 사실이 있던 대본 문장과 정렬된 STT 구간에 같은 종류·단위의 **다른 값**이 있음. 정렬 구간을 먼저 보고, 없으면 앞뒤 한 문장까지 봄. STT 의 한 값은 한 사실에만 씀 |
| `missing` | 어디에도 없음 |
| `unverified` | 규칙으로 못 찾은 **영문 이름**. STT 가 한글로 적었을 수 있어(`SeatFlow` → `시트플로우`) 누락으로 단정하지 않고 5장 LLM 이 확인 |

틀렸다가 바로 고쳐 말한 경우(`사십… 아니 사십이 퍼센트`)는 맞는 값이 있으므로 `matched` 입니다.

In [7]:
FactStatus = Literal["matched", "mismatched", "missing", "unverified", "sound_alike", "approximate"]
MismatchCause = Literal["speaker_error", "asr_error", "approximation"]


class FactCheck(BaseModel):
    fact_id: str
    type: str
    value: str = Field(description="대본 표기")
    normalized: str
    importance: Importance
    key_point_ids: list[str]
    status: FactStatus = Field(description="matched: 같은 값을 말함 / mismatched: 다른 값을 말함 / missing: 없음 / unverified: 규칙으로 판단 불가(LLM 확인)"
                                           " / sound_alike: 발음이 비슷한 다른 말로 나옴 — 판단 보류, 비율에서 뺌 / approximate: 값을 어림해 말함")
    stt_value: str | None = Field(default=None, description="STT 에서 찾은 표기 (다른 값이면 그 값)")
    stt_ids: list[int] = Field(default_factory=list)
    note: str = ""
    # 다른 값이 나왔을 때(mismatched)만 채운다 — 4-2 원인 신호
    stt_normalized: str | None = None
    stt_numeric_value: float | None = None
    stt_qualifier: str | None = None
    stt_span: tuple[int, int] | None = None
    signals: list[str] = Field(default_factory=list, description="원인 판단에 쓰는 규칙 신호")
    rule_cause: MismatchCause | None = Field(default=None, description="규칙 신호만으로 본 원인 (비교용)")
    sound: str | None = Field(default=None, description="두 수의 발음 관계: similar / near / swap / different / approx")
    cause: MismatchCause | None = Field(default=None, description="규칙으로 정한 원인 (발음이 비슷하면 정하지 않음)")
    cause_reason: str = ""


def _stt_sentence(norm: NormalizedScript, pos: int) -> int:
    return next((s.index for s in norm.sentences if s.start <= pos < s.end), 0)


def check_critical_facts(rubric: EvaluationRubric, stt_norm: NormalizedScript, alignments: list[Alignment]) -> list[FactCheck]:
    """Critical Fact Check (규칙): 대본의 수치·이름이 STT 에 그대로 나왔는지 본다.

    - 수치: STT 에서도 같은 수 파서로 읽어 정규형으로 비교한다 ('42%' = '사십이 퍼센트').
      같은 값이 없고, 대본 문장과 맞춰진 STT 구간(없으면 앞뒤 한 문장)에 같은 종류·단위의 다른 값이 있으면 mismatched
    - 이름: 공백·문장부호를 무시하고 찾는다. 영문 이름은 STT 가 한글로 받아 적을 수 있어('SeatFlow' → '시트플로우')
      못 찾으면 missing 이 아니라 unverified 로 두고 LLM 에 확인시킨다
    """
    stt_facts = extract_critical_facts(stt_norm)
    rubric_values = {(f.type, f.normalized) for f in rubric.critical_facts}
    region_of = {a.sentence_index: a.stt_ids for a in alignments if a.coverage > 0}
    stt_compact = _compact(stt_norm.text)[0]
    used: set[int] = set()  # '다른 값'으로 이미 쓴 STT 수치. 한 번 틀리게 말한 값이 여러 사실을 틀리게 만들지 않는다
    checks = []
    for fact in rubric.critical_facts:
        base = dict(fact_id=fact.id, type=fact.type, value=fact.value, normalized=fact.normalized,
                    importance=fact.importance, key_point_ids=fact.key_point_ids)
        if fact.type in NAME_TYPES:
            if _compact(fact.value)[0] in stt_compact:
                checks.append(FactCheck(status="matched", stt_value=fact.value, **base))
            elif re.search(r"[A-Za-z]", fact.value):
                checks.append(FactCheck(status="unverified", note="영문 이름: 한글 표기 여부를 LLM 이 확인", **base))
            else:
                checks.append(FactCheck(status="missing", **base))
            continue
        same = [f for f in stt_facts if f.type == fact.type and f.normalized == fact.normalized]
        if same:
            ids = sorted({_stt_sentence(stt_norm, s[0]) for f in same for s in f.spans})
            note = "한정어 차이" if fact.qualifier and fact.qualifier != same[0].qualifier else ""
            checks.append(FactCheck(status="matched", stt_value=same[0].value, stt_ids=ids, note=note, **base))
            continue
        # 같은 값이 없으면, 대본 문장과 맞춰진 STT 문장 → 그 앞뒤 한 문장 순서로 같은 종류·단위의 다른 값을 찾는다
        region = {j for i in fact.sentence_indices for j in region_of.get(i, [])}
        candidates = [
            k for k, f in enumerate(stt_facts)
            if k not in used and f.type == fact.type and f.unit == fact.unit and (f.type, f.normalized) not in rubric_values
        ]
        pick = None
        for area in (region, {j + d for j in region for d in (-1, 1)}):
            pick = next((k for k in candidates if any(_stt_sentence(stt_norm, s[0]) in area for s in stt_facts[k].spans)), None)
            if pick is not None:
                break
        if pick is not None:
            used.add(pick)
            other = stt_facts[pick]
            ids = sorted({_stt_sentence(stt_norm, s[0]) for s in other.spans})
            checks.append(FactCheck(status="mismatched", stt_value=other.value, stt_ids=ids,
                                    note=f"대본 {fact.value} ≠ STT {other.value}", stt_normalized=other.normalized,
                                    stt_numeric_value=other.numeric_value, stt_qualifier=other.qualifier,
                                    stt_span=tuple(other.spans[0]), **base))
        else:
            checks.append(FactCheck(status="missing", **base))
    return checks


display(pd.DataFrame([c.model_dump(include={"fact_id", "type", "value", "status", "stt_value", "note"})
                      for c in check_critical_facts(_rubric, sample_norm, align_sentences(_script_norm, _s_tokens, sample_norm, _t_tokens, _rubric.sentence_roles))]))

,fact_id,type,value,status,stt_value,note
0,CF1,proper_noun,SeatFlow,unverified,NaN,영문 이름: 한글 표기 여부를 LLM 이 확인
1,CF2,term,빈자리연구소,matched,빈자리연구소,
2,CF3,quantity,312명,matched,삼백십이 명,
3,CF4,duration,평균 23분,matched,평균 이십삼 분,


### 4-2. 다른 수치의 원인 신호 (규칙)

대본과 다른 수치가 나왔다고 모두 발표자 실수는 아닙니다. 두 수를 **소리 내어 읽은 형태**(`312` → `삼백십이`, 시각 `18:00` → `여섯시`)로 바꿔
발음 관계를 봅니다.

| 발음 관계 | 예 | 처리 |
|---|---|---|
| 어림 표현이 붙어 있고 방향·차이(30% 이내)가 맞음 | 42% → `40퍼센트 넘게`, 약 1,240만 건 → `천만 건이 넘는` | 어림 (`approximate`, 절반 인정) |
| 음절 순서만 바뀜 | 76% → `육십칠 퍼센트` | 발표자가 다르게 말함 (`mismatched`) — 음성 인식은 음절 순서를 바꾸지 않음 |
| 발음이 전혀 다른 수 | 600만 원 → `팔백만 원` | 발표자가 다르게 말함 (`mismatched`) |
| 발음이 비슷한 음절 하나 차이 · 음절 하나 빠짐/더해짐 | 312명 → `사백십이 명`, 42% → `40%` | **비슷한 수치** → 4-3 판단 보류 |
| 음절 하나가 조금 비슷함 (초성·중성·종성 중 두 개가 다름) | 오후 6시 → `오후 다섯 시` | **비슷한 수치** → 4-3 판단 보류 |

대본에 원래 있던 한정어(`약 18%` 의 `약`)는 발표자가 어림한 표현으로 보지 않습니다.

In [8]:
DIGIT_READING = "영일이삼사오육칠팔구"
# 어림 표현: 앞 값이 실제보다 작다(넘게) / 크다(가까이) / 방향 없음(약)
APPROX_LOWER = r"넘게|넘는|넘었|넘어|넘습|이상|남짓|조금 넘"
APPROX_UPPER = r"가까이|가까운|거의|안 되는|안되는|안 돼|못 미치|조금 안"
APPROX_ANY = r"약|대략|정도|쯤|가량|내외|안팎|얼추|대충"
PARTICLE = r"\s?(?:이|가|을|를|은|는|도|으로|로)?\s?"  # "천만 건이 넘는" 처럼 수와 어림 표현 사이의 조사
# 수 파서가 수치에 붙여 읽은 한정어 ("약 40%", "85% 이상")
QUALIFIER_LOWER, QUALIFIER_UPPER, QUALIFIER_ANY = {"이상", "초과", "남짓", "최소"}, {"이하", "미만", "이내", "거의", "최대"}, {"약", "대략", "가량", "정도", "내외"}


def _read_below_10000(n: int) -> str:
    out = ""
    for value, unit in ((1000, "천"), (100, "백"), (10, "십"), (1, "")):
        digit = n // value % 10
        if digit:
            out += ("" if digit == 1 and unit else DIGIT_READING[digit]) + unit
    return out


def read_number(text: str) -> str:
    """'42' → '사십이', '12400000' → '천이백사십만', '2.4' → '이점사'. 발표자가 소리 내어 읽는 방식."""
    integer, _, decimal = text.partition(".")
    n, parts = int(integer), []
    for value, unit in ((10**12, "조"), (10**8, "억"), (10**4, "만"), (1, "")):
        chunk = n // value % 10000
        if chunk:
            parts.append(("" if chunk == 1 and unit == "만" else _read_below_10000(chunk)) + unit)
    out = "".join(parts) or "영"
    return out + ("점" + "".join(DIGIT_READING[int(d)] for d in decimal) if decimal else "")


NATIVE_HOUR = {1: "한", 2: "두", 3: "세", 4: "네", 5: "다섯", 6: "여섯", 7: "일곱", 8: "여덟", 9: "아홉", 10: "열", 11: "열한", 12: "열두"}


def reading_of(normalized: str, kind: str = "") -> str:
    """정규형 안의 수를 말하는 대로 읽는다: '12,400,000건' → '천이백사십만', 시각 '18:30' → '여섯시 삼십분'."""
    if kind == "time" and re.fullmatch(r"\d{2}:\d{2}", normalized):
        hour, minute = map(int, normalized.split(":"))
        return NATIVE_HOUR[hour % 12 or 12] + "시" + (f" {read_number(str(minute))}분" if minute else "")
    return " ".join(read_number(g) for g in re.findall(r"\d+(?:\.\d+)?", normalized.replace(",", "")))


def jamo_diff(a: str, b: str) -> int:
    """한글 음절 두 개의 초성·중성·종성 중 다른 것의 수 (0~3). '삼'↔'사' = 1 (종성), '육'↔'팔' = 3."""
    if not ("가" <= a <= "힣" and "가" <= b <= "힣"):
        return 0 if a == b else 3
    x, y = ord(a) - 0xAC00, ord(b) - 0xAC00
    return sum(p != q for p, q in ((x // 588, y // 588), (x // 28 % 21, y // 28 % 21), (x % 28, y % 28)))


def mismatch_signals(check: FactCheck, script_fact: CriticalFact, stt_text: str) -> tuple[list[str], MismatchCause, str]:
    """대본 값과 STT 값이 다를 때, 원인을 가늠할 규칙 신호를 만든다. 최종 원인은 7장 LLM 교차 검증이 정한다.

    - 어림 표현: STT 값에 '넘게', '가까이', '약', '정도' 같은 말이 붙어 있고 방향과 차이가 맞으면 → 어림 (틀린 말 아님).
      대본에 원래 있던 한정어('약 18%' → '약 28%' 의 '약')는 발표자의 어림으로 보지 않는다
    - 음절 순서만 바뀜 ('칠십육' ↔ '육십칠'): 사람이 숫자를 바꿔 말하는 실수. 음성 인식은 음절 순서를 바꾸지 않는다
    - 발음이 비슷한 음절 하나 차이('삼백' ↔ '사백')나 음절 하나가 빠짐('사십이' → '사십'): 음성 인식 오류일 수 있음
    - 음절 하나가 조금 비슷함(초성·중성·종성 중 두 개가 다름, '여섯' ↔ '다섯'): 애매함 → 발표자 실수 쪽으로 두되 LLM 이 확인
    - 그 밖(다른 수): 발표자 실수 쪽
    세 번째 값(발음 관계)으로 LLM 인식 확인 대상을 고른다: similar·near 만 확인하고 swap·different·approx 는 규칙으로 정한다.
    """
    a = reading_of(check.normalized, check.type).replace(" ", "")
    b = reading_of(check.stt_normalized or "", check.type).replace(" ", "")
    signals = [f"읽기: 대본 '{a}' / STT '{b}'"]
    start, end = check.stt_span or (0, 0)
    before, after = stt_text[max(0, start - 6):start], stt_text[end:end + 8]
    t, script_value = check.stt_numeric_value, script_fact.numeric_value
    if script_value and t is not None:
        diff = abs(script_value - t) / abs(script_value)
        q = check.stt_qualifier if check.stt_qualifier != script_fact.qualifier else None
        lower = re.match(rf"{PARTICLE}(?:{APPROX_LOWER})", after) is not None or q in QUALIFIER_LOWER
        upper = (re.match(rf"{PARTICLE}(?:{APPROX_UPPER})", after) is not None or re.search(r"거의\s?$", before) is not None
                 or q in QUALIFIER_UPPER)
        loose = (re.search(rf"(?:{APPROX_ANY})\s?$", before) is not None or re.match(rf"{PARTICLE}(?:{APPROX_ANY})", after) is not None
                 or q in QUALIFIER_ANY)
        if lower or upper or loose:
            direction_ok = (lower and t <= script_value) or (upper and t >= script_value) or (loose and not lower and not upper)
            if direction_ok and diff <= 0.3:
                signals.append(f"어림 표현이 붙어 있고 방향이 맞음 (차이 {diff:.0%}): '{check.stt_value}{after.rstrip()}'")
                return signals, "approximation", "approx"
            signals.append(f"어림 표현이 붙어 있지만 방향이나 차이가 맞지 않음 (차이 {diff:.0%}) → 사실과 다른 말")
    if a and b and sorted(a) == sorted(b) and a != b:
        signals.append(f"음절 순서만 바뀜 ({a} ↔ {b}) → 사람이 숫자를 바꿔 말하는 실수에 가까움")
        return signals, "speaker_error", "swap"
    ops = [op for op in SequenceMatcher(None, a, b, autojunk=False).get_opcodes() if op[0] != "equal"]
    if len(ops) == 1:
        tag, i1, i2, j1, j2 = ops[0]
        if tag == "replace" and i2 - i1 == 1 and j2 - j1 == 1 and jamo_diff(a[i1], b[j1]) <= 1:
            signals.append(f"발음이 비슷한 음절 하나 차이 ({a[i1]} ↔ {b[j1]}) → 음성 인식 오류일 수 있음")
            return signals, "asr_error", "similar"
        if tag == "replace" and i2 - i1 == 1 and j2 - j1 == 1 and jamo_diff(a[i1], b[j1]) == 2:
            signals.append(f"발음이 조금 비슷한 음절 하나 차이 ({a[i1]} ↔ {b[j1]}) → 인식 오류인지 발표자 실수인지 애매함")
            return signals, "speaker_error", "near"
        if tag in ("delete", "insert") and max(i2 - i1, j2 - j1) == 1:
            changed = a[i1:i2] or b[j1:j2]
            signals.append(f"음절 하나('{changed}')가 {'빠짐' if tag == 'delete' else '더해짐'} → 음성 인식 오류일 수 있음")
            return signals, "asr_error", "similar"
    signals.append("발음이 비슷하지 않은 다른 수 → 발표자가 다른 값을 말했을 가능성이 큼")
    return signals, "speaker_error", "different"


def annotate_mismatches(fact_checks: list[FactCheck], rubric: EvaluationRubric, stt_norm: NormalizedScript) -> None:
    """mismatched 인 수치마다 원인 신호(signals)와 규칙 추정(rule_cause)을 붙인다."""
    facts = {f.id: f for f in rubric.critical_facts}
    for check in fact_checks:
        if check.status == "mismatched":
            check.signals, check.rule_cause, check.sound = mismatch_signals(check, facts[check.fact_id], stt_norm.text)


# 예시: 대본 표기와 STT 표기 짝 → 규칙 추정
for script_surface, stt_surface in [
    ("312명", "사백십이 명"), ("42%", "40%"), ("76%", "67퍼센트"), ("600만 원", "팔백만 원"), ("약 18%", "약 이십팔 퍼센트"),
    ("오후 6시", "오후 다섯 시"),
    ("약 1,240만 건", "천만 건이 넘는"), ("18명", "스무 명 가까이"), ("11.4%", "십일 퍼센트 정도"), ("42%", "오십 퍼센트 넘게"),
]:
    script_demo = [f for f in extract_critical_facts(normalize_script(script_surface)) if f.type not in NAME_TYPES][0]
    stt_demo = normalize_stt(stt_surface)
    found = [f for f in extract_critical_facts(stt_demo) if f.type not in NAME_TYPES][0]
    demo = FactCheck(fact_id="-", type=found.type, value=script_surface, normalized=script_demo.normalized, importance="normal",
                     key_point_ids=[], status="mismatched", stt_value=found.value, stt_normalized=found.normalized,
                     stt_numeric_value=found.numeric_value, stt_qualifier=found.qualifier, stt_span=tuple(found.spans[0]))
    signals, guess, sound = mismatch_signals(demo, script_demo, stt_demo.text)
    print(f"{script_surface:>11} → {stt_surface:<14} {guess:<14} {sound:<9} {signals[-1]}")

       312명 → 사백십이 명         asr_error      similar   발음이 비슷한 음절 하나 차이 (삼 ↔ 사) → 음성 인식 오류일 수 있음
        42% → 40%            asr_error      similar   음절 하나('이')가 빠짐 → 음성 인식 오류일 수 있음
        76% → 67퍼센트          speaker_error  swap      음절 순서만 바뀜 (칠십육 ↔ 육십칠) → 사람이 숫자를 바꿔 말하는 실수에 가까움
     600만 원 → 팔백만 원          speaker_error  different 발음이 비슷하지 않은 다른 수 → 발표자가 다른 값을 말했을 가능성이 큼
      약 18% → 약 이십팔 퍼센트      asr_error      similar   음절 하나('이')가 더해짐 → 음성 인식 오류일 수 있음
      오후 6시 → 오후 다섯 시        speaker_error  near      발음이 조금 비슷한 음절 하나 차이 (여 ↔ 다) → 인식 오류인지 발표자 실수인지 애매함
 약 1,240만 건 → 천만 건이 넘는       approximation  approx    어림 표현이 붙어 있고 방향이 맞음 (차이 19%): '천만 건이 넘는'
        18명 → 스무 명 가까이       approximation  approx    어림 표현이 붙어 있고 방향이 맞음 (차이 11%): '스무 명 가까이'
      11.4% → 십일 퍼센트 정도      approximation  approx    어림 표현이 붙어 있고 방향이 맞음 (차이 4%): '십일 퍼센트 정도'
        42% → 오십 퍼센트 넘게      speaker_error  different 발음이 비슷하지 않은 다른 수 → 발표자가 다른 값을 말했을 가능성이 큼


### 4-3. 비슷한 말 찾기 (규칙) — 판단 보류

대본과 STT 의 같은 자리에 **발음이 비슷한 다른 말**(단어·수치)이 나오면, 발표자가 잘못 말했는지 음성 인식이 잘못 적었는지 텍스트만으로는 가릴 수 없습니다.
그래서 이 평가에서는 판단하지 않고(**보류**), 점수 비율에서 빼고 개수만 남기며, 나중에 코칭 agent 가 판단할 수 있게 **위치와 함께 저장**합니다.

**찾는 곳**
1. 4장에서 대본과 다른 값이 나온 수치 중 4-2 에서 '비슷한 수치'로 본 것
2. 대본 문장과 정렬된 STT 문장을 형태소 단위로 맞춰(`SequenceMatcher`) **같은 자리에서 바뀐 표현**
   - 수치 → 비슷한 수치: 같은 수치를 다른 문장에서 맞게 말해 1에서 빠진 경우도 잡습니다 (`삼십 분` 을 한 번 맞게 말하고 다른 문장에서는 `사십 분` 으로 적힘)
   - 단어 → 단어: 대본 명사가 그 자리에서 빠지고, **발음 거리** 0.34 이하인 새 말이 나온 경우

**발음 거리** (`phonetic_distance`) — 음절 단위 편집 거리를 긴 쪽 음절 수로 나눈 값입니다. 음절을 바꾸는 비용은 초성·중성·종성 중 다른 수 / 3,
셋 다 다르면(발음이 전혀 다른 음절) 1.5 입니다. `재고`↔`제고` 0.17, `모델`↔`모텔` 0.17, `노쇼`↔`노조` 0.33 은 비슷한 말이고,
`월요일`↔`금요일` 0.5, `예측`↔`추천` 0.83 은 아닙니다 (발음이 비슷하지 않게 바꿔 말한 것은 5장 의미 평가가 봅니다).
명사에 붙은 접미사는 한 단어로 봅니다 (`이용`+`률` → `이용률`, 그래야 `이용료` 와 구별됩니다). 영문 이름의 한글 표기는 5장 이름 확인이 봅니다.

**저장하는 위치** (`SimilarItem`)

| 필드 | 뜻 |
|---|---|
| `script_sentence_index`, `script_span` | 대본 문장 번호와 정규화 대본 텍스트 안의 위치 |
| `stt_sentence_index`, `stt_span` | STT 문장 번호(T번호)와 정규화 STT 텍스트 안의 위치 |
| `stt_raw_span` | **원본 STT** 안의 위치 — 간투사를 지우기 전 텍스트라서, Deepgram 단어 타임스탬프와 맞춰 녹음 구간을 찾을 때 씁니다 |
| `key_point_ids`, `fact_id` | 이 자리가 걸린 Key Point 와 Critical Fact |
| `signals`, `rule_guess` | 규칙 신호(발음 거리, 읽기 비교)와 규칙만으로 본 원인 — 참고 |

원인은 이 평가에서 정하지 않습니다. 텍스트만으로는 LLM 도 규칙 이상으로 가리지 못했습니다 (가상 STT 에서 LLM 판단과 규칙 추정의 정확도가 같았음).
녹음이나 발표자 확인이 필요한 판단이라 코칭 agent 에게 넘깁니다.

**정리** (`settle_facts`) — 비슷한 말에 걸린 수치·이름은 `sound_alike`(판단 보류), 나머지 다른 수치는 4-2 규칙대로 `mismatched`(발표자가 다르게 말함) 또는 `approximate`(어림)입니다.

In [9]:
HintCause = Literal["asr_error", "speaker_error", "paraphrase"]
WORD_TAGS = {"NNG", "NNP", "SL", "SH", "XR"}   # 단어 후보는 대본 쪽 명사·어근·외국어만 본다
WORD_DISTANCE = 0.34                           # 발음 거리가 이 값 이하면 비슷한 단어 (2음절 단어에서 한 음절의 자모 두 개까지)
SIMILAR_SOUNDS = {"similar", "near"}           # 비슷한 수치로 보는 발음 관계 (4-2)
PARTICLE_TAIL = re.compile(r"(?:이랑|에서|으로|하고|까지|부터|은|는|이|가|을|를|의|에|로|도|만|과|와|랑)$")


class SimilarItem(BaseModel):
    """비슷한 말: 대본과 STT 의 같은 자리에 발음이 비슷한 다른 말(단어·수치)이 나온 곳.

    발표자가 잘못 말했는지 음성 인식이 잘못 적었는지는 텍스트만으로 가릴 수 없다. 그래서 점수 비율에서 빼고(판단 보류)
    개수만 점수에 넣으며, 위치와 함께 저장해 두고 나중에 코칭 agent 가 판단한다.
    """
    item_id: str
    kind: Literal["number", "word"]
    script_text: str = Field(description="대본 표현")
    stt_text: str = Field(description="STT 표현")
    script_sentence_index: int = Field(description="대본 문장 번호")
    script_span: tuple[int, int] = Field(description="정규화 대본 텍스트 기준 위치")
    stt_sentence_index: int = Field(description="정규화 STT 문장 번호 (T번호)")
    stt_span: tuple[int, int] = Field(description="정규화 STT 텍스트 기준 위치")
    stt_raw_span: tuple[int, int] | None = Field(description="원본 STT 텍스트 기준 위치 — Deepgram 단어 타임스탬프와 맞출 때 쓴다")
    fact_id: str | None = Field(default=None, description="관련 Critical Fact")
    key_point_ids: list[str] = Field(description="이 자리가 걸린 Key Point")
    signals: list[str] = Field(description="규칙 신호 (발음 거리, 읽기 비교 등)")
    rule_guess: HintCause = Field(description="규칙 신호만으로 본 원인 — 참고 (asr_error: 인식 오류 쪽 / speaker_error: 발표자 실수 쪽)")


def phonetic_distance(a: str, b: str) -> float:
    """음절 단위 편집 거리를 긴 쪽 음절 수로 나눈 값 (0 = 같음). 바꾸기 비용 = 초성·중성·종성 중 다른 수 / 3,
    셋 다 다른 음절(발음이 전혀 다름)은 1.5, 넣기·빼기 = 1.
    '재고'↔'제고' = 0.17 (중성 하나), '노쇼'↔'노조' = 0.33, '월요일'↔'금요일' = 0.5, '예측'↔'추천' = 0.83."""
    n, m = len(a), len(b)
    d = [[float(i + j) if i == 0 or j == 0 else 0.0 for j in range(m + 1)] for i in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diff = jamo_diff(a[i - 1], b[j - 1])
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + (1.5 if diff == 3 else diff / 3))
    return d[n][m] / max(n, m, 1)


def _plain(text: str) -> str:
    return re.sub(r"[^0-9A-Za-z가-힣]", "", text).lower()


def _stt_word(text: str, start: int, end: int) -> tuple[str, int, int]:
    """STT 쪽 어절(띄어쓰기 단위)에서 조사를 뗀 표현과 그 위치."""
    while start > 0 and not text[start - 1].isspace():
        start -= 1
    while end < len(text) and not text[end].isspace():
        end += 1
    word = PARTICLE_TAIL.sub("", text[start:end]) or text[start:end]
    return word, start, start + len(word)


def _contiguous(tokens: list[Token], text: str) -> bool:
    """형태소들이 붙어 있는가 (사이에 공백만). 떨어진 두 단어를 한 표현으로 묶지 않는다."""
    return all(not text[a.end:b.start].strip() for a, b in zip(tokens, tokens[1:]))


def locate_raw(raw: str, surface: str, norm_start: int, norm_length: int) -> tuple[int, int] | None:
    """정규화 STT 의 표현을 원본 STT 에서 찾는다 (간투사를 지워 위치가 달라지므로). 여러 번 나오면 상대 위치가 가장 가까운 것."""
    pattern = r"\s*".join(map(re.escape, surface.replace(" ", "")))
    found = [(m.start(), m.end()) for m in re.finditer(pattern, raw)] if pattern else []
    if not found:
        return None
    target = norm_start / max(norm_length, 1)
    return min(found, key=lambda span: abs(span[0] / max(len(raw), 1) - target))


def _number_signals(script_fact: CriticalFact, stt_fact: CriticalFact, stt_norm: NormalizedScript) -> tuple[list[str], str, str]:
    probe = FactCheck(fact_id=script_fact.id, type=script_fact.type, value=script_fact.value, normalized=script_fact.normalized,
                      importance=script_fact.importance, key_point_ids=script_fact.key_point_ids, status="mismatched",
                      stt_value=stt_fact.value, stt_normalized=stt_fact.normalized, stt_numeric_value=stt_fact.numeric_value,
                      stt_qualifier=stt_fact.qualifier, stt_span=tuple(stt_fact.spans[0]))
    return mismatch_signals(probe, script_fact, stt_norm.text)


def find_similar_items(rubric: EvaluationRubric, script_norm: NormalizedScript, script_tokens: list[Token],
                       stt_norm: NormalizedScript, stt_tokens: list[Token], alignments: list[Alignment],
                       fact_checks: list[FactCheck], raw_stt: str) -> list[SimilarItem]:
    """비슷한 말 찾기 (규칙).

    ① 4장에서 대본과 다른 값이 나온 수치 중 발음이 비슷하거나 애매한 것 (4-2 발음 관계 similar·near)
    ② 대본 문장과 정렬된 STT 문장을 형태소 단위로 맞춰(SequenceMatcher) 같은 자리에서 바뀐 표현
       - 수치 → 수치 (발음이 비슷한 것만): 같은 수치를 다른 문장에서 맞게 말해 ①에서 빠진 경우도 잡힌다
       - 단어 → 단어: 대본 명사가 그 자리에서 빠지고, 발음 거리 ≤ WORD_DISTANCE 인 새 말이 나온 경우
    발음이 전혀 다르거나 음절 순서만 바뀐 수치, 어림 표현은 여기 들지 않는다 — 4-2 규칙 원인으로 비율에 반영한다.
    """
    facts = {f.id: f for f in rubric.critical_facts}
    fact_by_value = {(f.type, f.normalized): f for f in rubric.critical_facts}
    stt_facts = {f.spans[0][0]: f for f in extract_critical_facts(stt_norm) if f.type in NUMERIC_TYPES}
    rubric_values = set(fact_by_value)
    items, seen = [], set()

    def stt_sentence_of(pos: int) -> int:
        return next((s.index for s in stt_norm.sentences if s.start <= pos < s.end), 0)

    def add(script_span, stt_span, fact_id, **fields):
        key = (_plain(fields["script_text"]), _plain(fields["stt_text"]))
        if key in seen:
            return
        seen.add(key)
        sentence = fields.pop("script_sentence_index")
        kp_ids = [kp.id for kp in rubric.key_points if sentence in kp.sentence_indices or (fact_id and fact_id in kp.fact_ids)]
        items.append(SimilarItem(
            item_id=f"R{len(items) + 1}", script_sentence_index=sentence, script_span=tuple(script_span),
            stt_sentence_index=stt_sentence_of(stt_span[0]), stt_span=tuple(stt_span),
            stt_raw_span=locate_raw(raw_stt, fields["stt_text"], stt_span[0], len(stt_norm.text)),
            fact_id=fact_id, key_point_ids=kp_ids, **fields))

    for c in fact_checks:  # ①
        if c.status == "mismatched" and c.sound in SIMILAR_SOUNDS:
            fact = facts[c.fact_id]
            sentence = min(fact.sentence_indices)
            span = next((sp for sp, i in zip(fact.spans, fact.sentence_indices) if i == sentence), fact.spans[0])
            add(span, c.stt_span, c.fact_id, kind="number", script_text=c.value, stt_text=c.stt_value,
                script_sentence_index=sentence, signals=c.signals, rule_guess=c.rule_cause)

    for a in alignments:  # ②
        if not a.stt_ids:
            continue
        src_all = [t for t in script_tokens if t.sentence == a.sentence_index]
        dst_all = [t for t in stt_tokens if t.sentence in a.stt_ids]
        sentence_plain = _plain(script_norm.sentences[a.sentence_index].text)
        matcher = SequenceMatcher(None, [t.key for t in src_all], [t.key for t in dst_all], autojunk=False)
        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            if tag != "replace" or i2 - i1 > 2 or j2 - j1 > 2:
                continue
            src, dst = src_all[i1:i2], dst_all[j1:j2]
            src_num = [t for t in src if t.tag == "NUM"]
            dst_num = [t for t in dst if t.tag == "NUM"]
            if len(src_num) == 1 and len(dst_num) == 1:
                s_type, s_value = src_num[0].key[1:-1].split(":", 1)
                d_type, d_value = dst_num[0].key[1:-1].split(":", 1)
                script_fact, stt_fact = fact_by_value.get((s_type, s_value)), stt_facts.get(dst_num[0].start)
                if (s_type != d_type or script_fact is None or stt_fact is None or (d_type, d_value) in rubric_values
                        or stt_fact.unit != script_fact.unit):
                    continue
                signals, guess, sound = _number_signals(script_fact, stt_fact, stt_norm)
                if sound in SIMILAR_SOUNDS:
                    add((src_num[0].start, src_num[0].end), (dst_num[0].start, dst_num[0].end), None, kind="number",
                        script_text=script_fact.value, stt_text=stt_fact.value, script_sentence_index=a.sentence_index,
                        signals=["같은 수치를 다른 곳에서 맞게 말했더라도 이 문장 자리에서는 다르게 나옴"] + signals, rule_guess=guess)
            elif (not src_num and not dst_num and all(t.tag in WORD_TAGS for t in src)
                  and _contiguous(src, script_norm.text) and _contiguous(dst, stt_norm.text)):
                script_word = script_norm.text[src[0].start:src[-1].end]
                sw = _plain(script_word)
                stt_id = stt_sentence_of(dst[0].start)
                if len(sw) < 2 or sw in _plain(stt_norm.sentences[stt_id].text) or re.search(r"[A-Za-z]", script_word):
                    continue  # 짧은 말, 그 STT 문장에 이미 있는 말, 영문 이름(한글 표기는 5장 이름 확인이 본다)은 제외
                options = [(stt_norm.text[dst[0].start:dst[-1].end], dst[0].start, dst[-1].end),
                           _stt_word(stt_norm.text, dst[0].start, dst[-1].end)]
                best, b_start, b_end = min(options, key=lambda o: phonetic_distance(sw, _plain(o[0])))
                dist = phonetic_distance(sw, _plain(best))
                if dist > WORD_DISTANCE or not _plain(best) or _plain(best) in sentence_plain:
                    continue
                fact = next((f for f in rubric.critical_facts if f.type in NAME_TYPES and a.sentence_index in f.sentence_indices
                             and (_plain(f.value) in sw or sw in _plain(f.value))), None)
                add((src[0].start, src[-1].end), (b_start, b_end), fact.id if fact else None, kind="word",
                    script_text=script_word, stt_text=best, script_sentence_index=a.sentence_index, rule_guess="asr_error",
                    signals=[f"발음 거리 {dist:.2f} ('{script_word}' ↔ '{best}')", "대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴"])
    return items


CAUSE_STATUS = {"speaker_error": "mismatched", "approximation": "approximate"}


def settle_facts(fact_checks: list[FactCheck], items: list[SimilarItem]) -> None:
    """사실 결과 정리 (규칙).
    - 발음이 비슷한 말에 걸린 수치·이름 → sound_alike (판단 보류, 비율에서 뺌)
    - 나머지 다른 수치 → 규칙 원인대로: 발음이 전혀 다름·순서만 바뀜 → mismatched, 어림 표현 → approximate
    """
    held = {it.fact_id for it in items if it.fact_id}
    for c in fact_checks:
        if c.fact_id in held and c.status in ("mismatched", "missing"):
            c.status, c.note = "sound_alike", "발음이 비슷한 다른 말로 나옴 — 판단 보류 (비율에서 뺌)"
        elif c.status == "mismatched":
            c.cause = c.rule_cause if c.rule_cause in CAUSE_STATUS else "speaker_error"
            c.cause_reason = {"speaker_error": "발음이 비슷하지 않거나 음절 순서만 바뀜 — 발표자가 다르게 말함",
                              "approximation": "어림 표현이 붙어 있고 방향이 맞음 — 어림해 말함"}[c.cause]
            c.status = CAUSE_STATUS[c.cause]


def without_spans(tokens: list[Token], spans: list[tuple[int, int]]) -> list[tuple[str, int]]:
    """주어진 위치와 겹치는 형태소를 뺀 (형태소, 문장 번호) 목록 — 대본 충실도에서 비슷한 말을 빼는 데 쓴다."""
    return [(t.key, t.sentence) for t in tokens if not any(t.start < end and start < t.end for start, end in spans)]


# 예시: 대본 두 문장과 STT
_demo_script = normalize_script("노쇼 비율은 18%에서 7%로 낮아졌습니다. 시범 운영은 30분 단위로 했습니다.")
_demo_raw = "음 노조 비율은 십팔 퍼센트에서 칠 퍼센트로 낮아졌습니다 어 시험 운영은 사십 분 단위로 했습니다"
_demo_stt = normalize_stt(_demo_raw)
_demo_s_tokens, _demo_t_tokens = positioned_tokens(_demo_script), positioned_tokens(_demo_stt)
_demo_rubric = EvaluationRubric.model_construct(critical_facts=extract_critical_facts(_demo_script), key_points=[])
for i, f in enumerate(_demo_rubric.critical_facts, 1):
    f.id = f"CF{i}"
_demo_align = align_sentences(_demo_script, [(t.key, t.sentence) for t in _demo_s_tokens], _demo_stt,
                              [(t.key, t.sentence) for t in _demo_t_tokens], ["evidence", "evidence"])
for item in find_similar_items(_demo_rubric, _demo_script, _demo_s_tokens, _demo_stt, _demo_t_tokens, _demo_align, [], _demo_raw):
    print(f"{item.item_id} [{item.kind}] {item.script_text} → {item.stt_text} | 대본 S{item.script_sentence_index} {item.script_span}"
          f" | STT T{item.stt_sentence_index} {item.stt_span} · 원본 {item.stt_raw_span} '{_demo_raw[slice(*item.stt_raw_span)]}'"
          f" | 규칙 추정 {item.rule_guess}")

R1 [word] 노쇼 → 노조 | 대본 S0 (0, 2) | STT T0 (0, 2) · 원본 (2, 4) '노조' | 규칙 추정 asr_error
R2 [word] 시범 → 시험 | 대본 S1 (25, 27) | STT T1 (30, 32) · 원본 (34, 36) '시험' | 규칙 추정 asr_error
R3 [number] 30분 → 사십 분 | 대본 S1 (32, 35) | STT T1 (37, 41) · 원본 (41, 45) '사십 분' | 규칙 추정 asr_error


## 5. LLM API 의미 평가 (슬라이드당 1회) — 대본 문장 단위

**대본 문장마다** 전달 여부를 판정합니다. 입력은 판정 대상 대본 문장(`[S번호]`, 문장 속 핵심 수치·이름)과 번호를 붙인 STT(`[T번호]`)입니다.
인사·전환 문장(`skip`)은 판정하지 않습니다.
- **왜 문장 단위인가** — 여러 문장으로 된 Key Point 를 통째로 판정하게 하면, 몇 문장을 말해야 '일부 전달'인지 LLM 이 매번 다르게 봅니다.
  문장 하나는 기준이 분명하고, 근거 STT 문장과 1:1 로 대응합니다. Key Point 판정은 6장에서 코드가 문장 판정을 모아 정합니다.
  대본 분석이 Key Point 를 어떻게 묶었는지에 판정이 덜 흔들리는 효과도 있습니다.
- **근거 → 이유 → 판정** 순서로 출력하게 해서, 판정을 먼저 정하고 근거를 끼워 맞추지 않게 합니다.
- 판정은 `said`(전달) / `partial`(일부) / `missing`(빠짐) / `contradicted`(모순), 확신도는 `high` / `medium` / `low` 입니다.
  문장에 적힌 핵심 수치·이름까지 말해야 `said` 이고, 표현만 다르고 뜻이 같으면 모순이 아닙니다.
- 4장에서 `unverified` 로 남은 영문 이름을 **같은 호출에서** 확인합니다 (추가 호출 없음).
- **규칙 결과는 입력에 넣지 않습니다.** LLM 이 규칙 결과에 끌려가지 않고 독립적으로 판단해야, 6장에서 둘을 비교해 어느 한쪽의 실수를 찾을 수 있습니다.

In [10]:
CoverageStatus = Literal["covered", "partial", "missing", "contradicted"]   # Key Point 판정 (코드가 문장 판정을 모아 정한다)
SentenceStatus = Literal["said", "partial", "missing", "contradicted"]     # 대본 문장 판정 (LLM)


class SentenceJudgment(BaseModel):
    # 근거 → 이유 → 판정 순서로 생성하게 해서, 근거를 먼저 찾고 판정하도록 한다
    sentence_id: int = Field(description="대본 문장 번호 [S번호]")
    evidence: list[int] = Field(description="이 문장의 내용이 나온 STT 문장 번호 [T번호]. missing 이면 빈 목록")
    reason: str = Field(description="판단 이유 한 문장. contradicted 면 대본과 무엇이 다른지")
    status: SentenceStatus
    confidence: Literal["high", "medium", "low"]


class NameCheck(BaseModel):
    name_id: str = Field(description="이름 확인 목록의 번호 (N1, N2 …)")
    said: bool = Field(description="발표자가 이 이름을 말했는가 (한글로 받아 적혔어도 말한 것)")
    evidence: list[int] = Field(description="말한 STT 문장 번호")


class SemanticEvaluation(BaseModel):
    sentences: list[SentenceJudgment] = Field(description="판정 대상 대본 문장마다 하나씩")
    name_checks: list[NameCheck] = Field(description="이름 확인 목록마다 하나씩")


SEMANTIC_EVAL_PROMPT = """
너는 발표 코칭 서비스의 채점자야. 슬라이드 한 장에 대해, 발표자가 실제로 말한 내용(STT)이 대본의 각 문장을 전달했는지 판단해.

# STT 의 특성 — 아래 차이는 전달 실패로 보지 마
- 음성 인식 결과라 띄어쓰기·맞춤법이 틀리고 문장부호가 거의 없어.
- 숫자가 한글로 적힐 수 있어 ("사십이 퍼센트" = 42%, "이천이십오 년 삼 월" = 2025년 3월).
- 영문 이름이 발음대로 한글로 적힐 수 있어 ("시트플로우" = SeatFlow).
- 발표자는 문장을 합치거나 쪼개거나 순서를 바꿔 말해. STT 한 문장이 여러 대본 문장의 근거가 될 수 있어.

# 대본 문장 판정
판정 대상 문장마다 하나씩, 다른 문장과 따로 판단해.
- evidence: 먼저 이 문장의 내용이 나온 STT 문장 번호를 적어. 없으면 빈 목록.
- reason: 판단 이유를 한 문장으로. contradicted 면 대본과 무엇이 다른지 적어.
- status:
  - said: 문장의 뜻이 전달됨. 표현·어순이 달라도 되지만, 문장에 적힌 핵심 수치·이름은 말해야 해.
  - partial: 일부만 전달됨. 핵심 수치·이름을 빼고 말했거나("42% 줄었다" 대신 "많이 줄었다"), 문장 속 여러 내용 중 일부만 말함.
  - missing: 이 문장의 내용이 STT 에 없음.
  - contradicted: 대본과 다른 수치를 말했거나 대본과 반대되는 내용을 말함. 표현만 다르고 뜻이 같으면 contradicted 가 아니야.
- confidence: 판단이 확실하면 high, 해석의 여지가 있으면 medium, 근거가 약하면 low.

# 이름 확인
- 목록의 이름을 발표자가 말했는지 판단해. 발음대로 한글로 적혔어도 말한 거야.
"""


def eval_config_hash() -> str:
    payload = json.dumps({"model": MODEL, "prompt": SEMANTIC_EVAL_PROMPT, "schema": SemanticEvaluation.model_json_schema()},
                         ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


def build_eval_llm():
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, model=MODEL, max_retries=2).with_structured_output(SemanticEvaluation)


def judged_sentences(rubric: EvaluationRubric) -> list[int]:
    """판정 대상 대본 문장: 인사·전환(skip)이 아닌 문장과 Key Point 근거 문장."""
    in_key_points = {i for kp in rubric.key_points for i in kp.sentence_indices}
    return [i for i, role in enumerate(rubric.sentence_roles) if role != "skip" or i in in_key_points]


def semantic_eval_message(slide_number: int, rubric: EvaluationRubric, stt_norm: NormalizedScript,
                          unverified: list[FactCheck]) -> str:
    """입력: 판정 대상 대본 문장(번호 + 문장 속 핵심 수치·이름) + 번호를 붙인 STT.

    규칙 쪽 판정(정렬 점수, 사실 검증 결과)은 넣지 않는다. LLM 이 독립적으로 판단해야 둘을 비교해 충돌을 찾을 수 있다.
    """
    lines = [f"[슬라이드 {slide_number}]", "## 대본 문장 (판정 대상)"]
    for i in judged_sentences(rubric):
        key_facts = [f.value for f in rubric.critical_facts if i in f.sentence_indices]
        lines.append(f"[S{i}] {rubric.sentences[i].text}" + (f" (핵심 수치·이름: {', '.join(key_facts)})" if key_facts else ""))
    if unverified:
        lines += ["", "## 이름 확인"]
        lines += [f"- N{i} {check.value}" for i, check in enumerate(unverified, 1)]
    lines += ["", "## 발표 STT"]
    lines += [f"[T{s.index}] {s.text}" for s in stt_norm.sentences] or ["(발화 없음)"]
    return "\n".join(lines)


def evaluate_semantics(message: str, eval_llm) -> SemanticEvaluation:
    """LLM API (의미 평가): 대본 문장별 전달 여부 + 이름 확인을 한 번의 호출로 받는다."""
    return eval_llm.invoke([("system", SEMANTIC_EVAL_PROMPT), ("user", message)])


print("eval_config_hash:", eval_config_hash())
print(semantic_eval_message(1, _rubric, sample_norm, []))

eval_config_hash: faaab35fc2b9
[슬라이드 1]
## 대본 문장 (판정 대상)
[S1] 도서관 좌석 혼잡도를 예측해 빈자리를 미리 알려 주는 서비스 SeatFlow를 만든 빈자리연구소입니다. (핵심 수치·이름: SeatFlow, 빈자리연구소)
[S2] 시험 기간이 되면 열람실은 늘 붐빕니다.
[S3] 저희가 대학생 312명에게 물어보니, 빈자리를 찾느라 하루 평균 23분을 쓴다고 답했습니다. (핵심 수치·이름: 312명, 평균 23분)
[S4] 자리가 없는 것만이 문제가 아닙니다.
[S5] 가방만 두고 자리를 비우는 경우가 많아서, 실제로는 비어 있는 좌석도 찾기 어렵습니다.
[S6] 그래서 저희는 좌석이 언제 비는지 미리 알려 주면 이 시간을 줄일 수 있다고 생각했습니다.

## 발표 STT
[T0] 안녕하십니까
[T1] 도서관 좌석 혼잡도를 예측해서 빈자리를 미리 알려주는 서비스 시트플로우를 만든 빈자리 연구소입니다.
[T2] 시험 기간만 되면 열람실이 항상 사람으로 꽉 차죠
[T3] 저희가 대학생 삼백십이 명에게 물어보니까 빈자리를 찾느라 하루 평균 이십삼 분을 쓴다고 답했습니다
[T4] 자리가 없는 것만 문제가 아닙니다
[T5] 가방만 두고 자리를 비우는 경우가 많아서 실제로는 비어있는 좌석도 찾기가 어렵습니다
[T6] 그래서 저희는 이제 좌석이 언제 비는지 미리 알려 주면 이 시간을 줄일 수 있다고 생각했습니다


## 6. 문장별 병합 · 충돌 검사 → Key Point 판정 (코드)

**문장마다** LLM 판정과 규칙 결과(3장 단어 비율, 4장 수치·이름, 4-3 비슷한 말)를 합칩니다. STT 에 없는 근거 번호는 버리고, 아래 경우를 **충돌**로 표시합니다.
충돌이 없으면 LLM 판정을 그대로 씁니다.

| 충돌 | 조건 (문장 하나 기준) |
|---|---|
| `contradicted` | 모순 판정은 점수를 깎으므로 항상 다시 확인 |
| `low_confidence` | LLM 확신도가 low |
| `evidence_invalid` | missing 이 아닌데 유효한 근거 문장이 없음 |
| `fact_missing` | said 인데 이 문장의 수치·이름이 STT 에 없음 |
| `sound_alike` | 이 문장 자리에 비슷한 말(4-3)이 있는데 said 가 아님 — 그 차이는 빼고(대본대로 말한 것으로 보고) 다시 판정 |
| `speaker_changed` | 발음이 전혀 다른 수치를 말했는데(4-2) contradicted 가 아님 |
| `approximation` | 수치를 어림해 말했는데(4-2) said |
| `fact_present` | missing 인데 이 문장에만 있는 수치를 STT 에서 찾음 |
| `lexical_present` | missing 인데 이 문장의 단어가 60% 이상 나옴 |
| `lexical_absent` | said 인데 이 문장의 단어가 20% 도 안 나옴 |

기준값은 가상 STT 에서 본 분포로 정했습니다: 빠뜨린 문장의 단어 비율은 최대 0.5, 말한 문장(그대로·의역)은 한 문장을 빼면 0.2 이상이었습니다.

**Key Point 판정 = 근거 문장 판정 모으기** (`aggregate_status`, 교차 검증 뒤에 계산)

| 근거 문장 판정 | Key Point 판정 |
|---|---|
| 하나라도 contradicted | contradicted |
| 모두 said | covered |
| 모두 missing | missing |
| 그 밖 | partial |

LLM 1차 문장 판정을 모은 값(`first_status`)도 함께 남겨, 교차 검증 전후를 비교합니다.

In [11]:
class SentenceResult(BaseModel):
    """대본 문장 한 개의 판정. LLM 판정 + 규칙 결과 + 충돌."""
    sentence_index: int
    text: str
    role: str
    status: SentenceStatus = Field(description="최종 판정")
    first_status: SentenceStatus = Field(description="LLM 1차 판정")
    confidence: str
    evidence: list[int] = Field(description="근거 STT 문장 번호 (범위 밖 번호는 버림)")
    reason: str
    lexical_coverage: float = Field(description="이 문장의 내용 형태소가 STT 정렬 구간에 나온 비율 (3장)")
    fact_statuses: dict[str, str] = Field(description="이 문장에 있는 사실별 규칙 검증 결과")
    similar_ids: list[str] = Field(default_factory=list, description="이 문장 자리의 비슷한 말 (R번호, 판단 보류)")
    conflicts: list[str] = Field(default_factory=list, description="규칙 결과와 LLM 판정이 어긋난 이유")
    verified: bool = False


class KeyPointResult(BaseModel):
    """Key Point 판정 = 근거 문장 판정을 모은 것 (코드, aggregate_status)."""
    key_point_id: str
    content: str
    importance: Importance
    sentence_indices: list[int]
    status: CoverageStatus = Field(description="최종 판정")
    first_status: CoverageStatus = Field(description="LLM 1차 문장 판정을 모은 판정")
    evidence: list[int]
    evidence_text: str
    reason: str
    lexical_coverage: float
    fact_statuses: dict[str, str] = Field(description="이 Key Point 에 연결된 사실별 규칙 검증 결과")
    similar_ids: list[str] = Field(default_factory=list)
    conflicts: list[str] = Field(default_factory=list, description="근거 문장들의 충돌 이유")
    verified: bool = False


# 충돌로 볼 기준. 가상 STT 에서 빠뜨린 문장의 단어 비율은 최대 0.5, 말한 문장(그대로·의역)은 한 문장을 빼면 0.2 이상이었다
LEXICAL_PRESENT = 0.6   # LLM 이 missing 이라는데 대본 단어가 이만큼 나왔으면 의심
LEXICAL_ABSENT = 0.2    # LLM 이 said 라는데 대본 단어가 이만큼도 안 나왔으면 의심

CONFLICT_TEXT = {
    "contradicted": "모순 판정은 항상 다시 확인",
    "low_confidence": "LLM 확신도가 낮음",
    "evidence_invalid": "전달됐다고 했는데 근거 STT 문장이 없음",
    "fact_missing": "said 인데 이 문장의 핵심 수치·이름이 STT 에 없음",
    "sound_alike": "발음이 비슷한 다른 말이 있는데 said 가 아님 — 그 차이는 빼고(대본대로 말한 것으로 보고) 다시 판정",
    "speaker_changed": "발음이 전혀 다른 수치를 말했는데 contradicted 가 아님",
    "approximation": "수치를 어림해 말했는데 said",
    "fact_present": "missing 인데 이 문장에만 있는 핵심 수치를 STT 에서 찾음",
    "lexical_present": "missing 인데 이 문장의 단어가 STT 에 많이 나옴",
    "lexical_absent": "said 인데 이 문장의 단어가 STT 에 거의 없음",
}


def aggregate_status(statuses: list[str]) -> str:
    """문장 판정 → Key Point 판정: 모순이 하나라도 있으면 contradicted, 모두 said 면 covered, 모두 missing 이면 missing, 나머지 partial."""
    if not statuses:
        return "missing"
    if "contradicted" in statuses:
        return "contradicted"
    if all(s == "said" for s in statuses):
        return "covered"
    if all(s == "missing" for s in statuses):
        return "missing"
    return "partial"


def resolve_name_checks(fact_checks: list[FactCheck], unverified: list[FactCheck], semantic: SemanticEvaluation) -> None:
    """규칙으로 못 찾은 영문 이름(unverified)을 LLM 이름 확인 결과로 확정한다."""
    said = {c.name_id: c for c in semantic.name_checks}
    for i, check in enumerate(unverified, 1):
        result = said.get(f"N{i}")
        if result is not None and result.said:
            check.status, check.stt_ids, check.note = "matched", result.evidence, "LLM 확인: 한글 표기 등으로 말함"
        else:
            check.status, check.note = "missing", "LLM 확인: 말하지 않음"


def merge_sentences(rubric: EvaluationRubric, semantic: SemanticEvaluation, fact_checks: list[FactCheck],
                    alignments: list[Alignment], stt_norm: NormalizedScript, items: list[SimilarItem]) -> list[SentenceResult]:
    """Result Merger + Conflict Check (코드): 대본 문장마다 LLM 판정과 규칙 결과를 합치고, 어긋나면 이유를 남긴다."""
    judgments = {j.sentence_id: j for j in semantic.sentences}
    coverage = {a.sentence_index: a.coverage for a in alignments}
    facts = {f.id: f for f in rubric.critical_facts}
    n_stt = len(stt_norm.sentences)
    results = []
    for i in judged_sentences(rubric):
        j = judgments.get(i) or SentenceJudgment(sentence_id=i, evidence=[], reason="LLM 판정 누락", status="missing", confidence="low")
        evidence = sorted({t for t in j.evidence if 0 <= t < n_stt})
        lexical = coverage.get(i, 0.0)
        checks = {c.fact_id: c for c in fact_checks if i in facts[c.fact_id].sentence_indices}
        statuses = {fid: c.status for fid, c in checks.items()}
        similar = [it for it in items if it.script_sentence_index == i]

        conflicts = []
        if j.status == "contradicted":
            conflicts.append("contradicted")
        if j.confidence == "low":
            conflicts.append("low_confidence")
        if j.status != "missing" and not evidence:
            conflicts.append("evidence_invalid")
        if j.status == "said" and "missing" in statuses.values():
            conflicts.append("fact_missing")
        # 비슷한 말(4-3)은 판단 보류: 그 차이 때문에 낮게 판정했을 수 있으면 빼고 다시 판정한다
        if similar and j.status != "said":
            conflicts.append("sound_alike")
        if "mismatched" in statuses.values() and j.status != "contradicted":
            conflicts.append("speaker_changed")
        if "approximate" in statuses.values() and j.status == "said":
            conflicts.append("approximation")
        # 여러 문장에 나오는 수치는 다른 문장을 말하면서 나왔을 수 있어서, 이 문장에만 있는 수치만 본다
        if j.status == "missing" and any(c.status == "matched" and c.type in NUMERIC_TYPES and facts[c.fact_id].sentence_indices == [i]
                                         for c in checks.values()):
            conflicts.append("fact_present")
        if j.status == "missing" and lexical >= LEXICAL_PRESENT:
            conflicts.append("lexical_present")
        if j.status == "said" and lexical < LEXICAL_ABSENT:
            conflicts.append("lexical_absent")

        results.append(SentenceResult(
            sentence_index=i, text=rubric.sentences[i].text, role=rubric.sentence_roles[i],
            status=j.status, first_status=j.status, confidence=j.confidence, evidence=evidence, reason=j.reason,
            lexical_coverage=round(lexical, 3), fact_statuses=statuses, similar_ids=[it.item_id for it in similar],
            conflicts=conflicts,
        ))
    return results


def aggregate_key_points(rubric: EvaluationRubric, sentences: list[SentenceResult], fact_checks: list[FactCheck],
                         items: list[SimilarItem], stt_norm: NormalizedScript) -> list[KeyPointResult]:
    """Key Point 판정 = 근거 문장 판정을 모은 것. 1차(LLM 문장 판정)와 최종(교차 검증 반영)을 각각 모은다."""
    by_index = {s.sentence_index: s for s in sentences}
    checks = {c.fact_id: c for c in fact_checks}
    results = []
    for kp in rubric.key_points:
        own = [by_index[i] for i in kp.sentence_indices if i in by_index]
        evidence = sorted({t for s in own for t in s.evidence})
        not_said = [f"S{s.sentence_index} {s.status}: {s.reason}" for s in own if s.status != "said"]
        results.append(KeyPointResult(
            key_point_id=kp.id, content=kp.content, importance=kp.importance, sentence_indices=kp.sentence_indices,
            status=aggregate_status([s.status for s in own]), first_status=aggregate_status([s.first_status for s in own]),
            evidence=evidence, evidence_text=" ".join(stt_norm.sentences[t].text for t in evidence),
            reason=" / ".join(not_said) or "근거 문장을 모두 전달함",
            lexical_coverage=round(sum(s.lexical_coverage for s in own) / max(len(own), 1), 3),
            fact_statuses={fid: checks[fid].status for fid in kp.fact_ids if fid in checks},
            similar_ids=[it.item_id for it in items if kp.id in it.key_point_ids],
            conflicts=sorted({c for s in own for c in s.conflicts}), verified=any(s.verified for s in own),
        ))
    return results

## 7. LLM API 교차 검증 (충돌한 문장이 있는 슬라이드만, 1회)

충돌한 **문장**만 모아 슬라이드마다 한 번에 다시 판정합니다. 판단에 필요한 정보를 모두 보여 줍니다.
- 대본 문장과 그 문장이 속한 Key Point
- 1차 판정·확신도·이유
- 관련 STT 문장: LLM 근거 ∪ 규칙 정렬 구간, 앞뒤 한 문장씩 더
- 이 문장의 수치·이름 확인 결과, 단어 비율, 충돌 이유
- **비슷한 말**(4-3) — 판정에서 빼라고 알려 줍니다. 발표자가 대본대로 말한 것으로 보고 나머지 내용으로 판정하게 합니다.
  발음이 전혀 다른 수치를 말한 핵심 수치는 contradicted, 어림한 핵심 수치는 최대 partial 입니다.

규칙도 틀릴 수 있다는 점(한글 숫자, 한글로 적힌 이름, 고쳐 말하기, 의역이면 단어 비율이 낮음)을 알려 주고,
제시한 STT 문장에서 확인되는 것만 근거로 삼게 합니다. 충돌이 없는 슬라이드는 호출하지 않습니다.

In [12]:
class VerifiedSentence(BaseModel):
    sentence_id: int
    reason: str = Field(description="최종 판단 이유 한 문장")
    status: SentenceStatus


class VerifierResult(BaseModel):
    judgments: list[VerifiedSentence] = Field(description="검증 대상 문장마다 하나씩")


VERIFIER_PROMPT = """
너는 발표 채점 검증자야. 1차 판정과 규칙 기반 확인 결과가 어긋난 대본 문장만 다시 판단해 최종 판정을 내려.

# 판정 기준 (문장 하나 기준)
- said: 문장의 뜻이 전달됨. 표현·어순이 달라도 되지만, 문장에 적힌 핵심 수치·이름은 말해야 해.
- partial: 일부만 전달됨. 핵심 수치·이름이 빠졌거나 문장 속 여러 내용 중 일부만 말함.
- missing: 이 문장의 내용이 STT 에 없음.
- contradicted: 대본과 다른 수치를 말했거나 대본과 반대되는 내용을 말함. 표현만 다르고 뜻이 같으면 contradicted 가 아니야.

# 발음이 비슷한 말 (판정에서 뺌)
대본과 STT 의 같은 자리에 발음이 비슷한 다른 말이 나온 곳은, 발표자가 잘못 말했는지 음성 인식이 잘못 적었는지 알 수 없어서
나중에 따로 확인해. 그 차이는 판정에 쓰지 말고, 발표자가 대본대로 말한 것으로 보고 나머지 내용으로 판정해.

# 수치
- 발음이 전혀 다른 수치를 말했으면(규칙 확인의 '발표자가 다르게 말함'), 그 문장의 핵심 수치면 contradicted.
- 수치를 어림해 말했으면(사실이지만 정확한 값은 아님), 그 문장의 핵심 수치면 최대 partial.

# 주의
- 규칙 기반 확인도 틀릴 수 있어. 음성 인식이 숫자를 한글로 적었거나("사십이 퍼센트"), 영문 이름을 한글로 적었거나("시트플로우"),
  발표자가 틀린 숫자를 말했다가 바로 고쳐 말한 경우("사십… 아니 사십이 퍼센트")에는 규칙이 불일치로 볼 수 있어. 이런 경우는 전달로 봐.
- 대본 단어가 STT 에 나온 비율은 참고만 해. 다른 말로 바꿔 말해도(의역) 뜻이 같으면 said 야.
- 제시한 STT 문장에서 직접 확인되는 것만 근거로 삼아.
"""


def verifier_config_hash() -> str:
    payload = json.dumps({"model": MODEL, "prompt": VERIFIER_PROMPT, "schema": VerifierResult.model_json_schema()},
                         ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


def build_verifier_llm():
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, model=MODEL, max_retries=2).with_structured_output(VerifierResult)


def _stt_window(ids, stt_norm: NormalizedScript) -> list[str]:
    """번호들과 앞뒤 한 문장씩."""
    ids = sorted({k for j in ids for k in (j - 1, j, j + 1) if 0 <= k < len(stt_norm.sentences)})
    return [f"  [T{j}] {stt_norm.sentences[j].text}" for j in ids] or ["  (없음)"]


FACT_DETAIL = {
    "matched": "STT 에 있음 ({stt})", "mismatched": "STT 에는 {stt} (발음이 전혀 다름 — 발표자가 다르게 말함)",
    "missing": "STT 에 없음", "unverified": "확인 못 함", "sound_alike": "STT 에는 발음이 비슷한 {stt} (판정에서 뺌)",
    "approximate": "STT 에는 {stt} (어림해 말함)",
}


def verifier_message(rubric: EvaluationRubric, sentences: list[SentenceResult], fact_checks: list[FactCheck],
                     alignments: list[Alignment], stt_norm: NormalizedScript, items: list[SimilarItem]) -> str:
    """검증 입력: 충돌한 문장만, 필요한 정보만 (대본 문장, 속한 Key Point, 1차 판정, 관련 STT 문장, 규칙 확인, 비슷한 말, 충돌 이유)."""
    checks = {c.fact_id: c for c in fact_checks}
    aligned = {a.sentence_index: a.stt_ids for a in alignments}
    by_id = {it.item_id: it for it in items}
    lines = ["# 검증 대상"]
    for s in sentences:
        if not s.conflicts:
            continue
        kps = [kp for kp in rubric.key_points if s.sentence_index in kp.sentence_indices]
        # 관련 STT 문장 = LLM 근거 ∪ 규칙 정렬 구간, 앞뒤 한 문장씩 더
        ids = set(s.evidence) | set(aligned.get(s.sentence_index, []))
        lines += [f"## S{s.sentence_index} \"{s.text}\""]
        lines += [f"속한 Key Point: {kp.id} [{kp.importance}] {kp.content}" for kp in kps]
        lines += [f"1차 판정: {s.first_status} (확신도 {s.confidence}) — {s.reason}", "관련 STT 문장:"]
        lines += _stt_window(ids, stt_norm)
        fact_lines = [f"  {checks[fid].value}: " + FACT_DETAIL[status].format(stt=checks[fid].stt_value or "다른 말")
                      for fid, status in s.fact_statuses.items()]
        lines += ["규칙 기반 확인:"] + (fact_lines or ["  (핵심 수치·이름 없음)"])
        similar = [by_id[i] for i in s.similar_ids if i in by_id]
        if similar:
            lines.append("발음이 비슷한 말 (판정에서 뺌):")
            lines += [f"  대본 '{it.script_text}' → STT '{it.stt_text}' [T{it.stt_sentence_index}]" for it in similar]
        lines.append(f"이 문장의 단어가 STT 에 나온 비율: {s.lexical_coverage:.0%}")
        lines.append("충돌 이유: " + "; ".join(CONFLICT_TEXT[c] for c in s.conflicts))
        lines.append("")
    return "\n".join(lines)


def verify(message: str, verifier_llm) -> VerifierResult:
    """LLM API (교차 검증): 충돌한 문장만 다시 판정한다. 슬라이드마다 한 번에 묶어 부른다."""
    return verifier_llm.invoke([("system", VERIFIER_PROMPT), ("user", message)])


def apply_verification(sentences: list[SentenceResult], verified: VerifierResult | None) -> None:
    if verified is None:
        return
    by_id = {v.sentence_id: v for v in verified.judgments}
    for s in sentences:
        v = by_id.get(s.sentence_index)
        if s.conflicts and v is not None:
            s.status, s.reason, s.verified = v.status, f"[검증] {v.reason}", True


print("verifier_config_hash:", verifier_config_hash())

verifier_config_hash: 57b9dfb40673


## 8. 점수 계산 (코드)

LLM 은 점수를 매기지 않습니다. 판정과 규칙 결과로 코드가 계산합니다. **비슷한 말(4-3)은 비율에서 빼고 개수로만 넣습니다.**

| 점수 | 계산 |
|---|---|
| Content Coverage | Σ(Key Point 판정 점수 × 중요도 가중치) / Σ(가중치). 판정 점수 covered 1 / partial 0.5 / missing 0 / contradicted −0.5, 가중치 critical 3 / high 2 / normal 1 (대본 분석 노트북의 `SCORE_WEIGHT`). 음수면 0. Key Point 판정은 문장 판정을 모은 것이고, 비슷한 말의 차이는 판정에 쓰지 않음 (6·7장) |
| Critical Fact Accuracy | Σ(사실 점수 × 가중치) / Σ(가중치). 맞게 말함 1, 어림해 말함 0.5, 빠짐·발표자가 다르게 말함 0. **비슷한 말에 걸린 수치·이름(`sound_alike`)은 분자·분모에서 뺌**. 사실이 없는 슬라이드는 `None` |
| Script Fidelity | 3장의 대본 충실도. **비슷한 말 자리의 형태소는 대본·STT 양쪽에서 뺌** |
| `similar_words` / `similar_numbers` | 비율에서 뺀 비슷한 단어·수치의 **개수**. 판단은 나중에 코칭 agent 가 합니다 |

틀린 말(contradicted)은 빠뜨린 것(missing)보다 낮게 봅니다. 청중에게 잘못된 정보를 준 것이기 때문입니다.
발표 전체 점수는 슬라이드 점수의 가중 평균이고(각각 Key Point 가중치 합 / 사실 가중치 합 / 대본 형태소 수로 가중), 개수는 합입니다.

In [13]:
STATUS_SCORE = {"covered": 1.0, "partial": 0.5, "missing": 0.0, "contradicted": -0.5}
FACT_SCORE = {"matched": 1.0, "approximate": 0.5}  # 나머지(missing / mismatched)는 0, sound_alike 는 계산에서 뺀다
# 중요도 가중치 SCORE_WEIGHT (critical 3 / high 2 / normal 1) 는 대본 분석 노트북 5장에서 정의한 값을 그대로 쓴다


def slide_scores(results: list[KeyPointResult], fact_checks: list[FactCheck], fidelity: dict, items: list[SimilarItem]) -> dict:
    """Score Engine (코드): LLM 은 점수를 매기지 않는다. 판정과 규칙 결과로 점수를 계산한다.

    - content_coverage = Σ(판정 점수 × 중요도 가중치) / Σ(중요도 가중치). 모순(-0.5) 때문에 음수가 되면 0
    - critical_fact_accuracy = Σ(사실 점수 × 가중치) / Σ(가중치). 맞게 말함 1, 어림해 말함 0.5, 빠짐·틀림 0.
      발음이 비슷한 다른 말로 나온 수치·이름(sound_alike)은 판단을 보류하므로 분자·분모에서 뺀다 (사실이 없으면 None)
    - script_fidelity = 대본 내용 형태소가 같은 순서로 나온 정도 (3장). 비슷한 말 자리의 형태소는 양쪽에서 뺀다
    - similar_words / similar_numbers = 비율에서 뺀 비슷한 말의 개수. 판단은 나중에 코칭 agent 가 한다
    """
    weight = sum(SCORE_WEIGHT[r.importance] for r in results)
    coverage = sum(STATUS_SCORE[r.status] * SCORE_WEIGHT[r.importance] for r in results) / weight if weight else 0.0
    counted = [c for c in fact_checks if c.status != "sound_alike"]
    fact_weight = sum(SCORE_WEIGHT[c.importance] for c in counted)
    fact_hit = sum(SCORE_WEIGHT[c.importance] * FACT_SCORE.get(c.status, 0.0) for c in counted)
    return {
        "content_coverage": round(max(coverage, 0.0), 3),
        "critical_fact_accuracy": round(fact_hit / fact_weight, 3) if fact_weight else None,
        "script_fidelity": fidelity["fidelity"],
        "similar_words": sum(it.kind == "word" for it in items),
        "similar_numbers": sum(it.kind == "number" for it in items),
        "_weights": {"key_points": weight, "facts": fact_weight, "script_tokens": fidelity.get("script_tokens", 0)},
    }


def take_scores(slide_evaluations: list) -> dict:
    """연습 한 번(발표 전체) 점수 = 슬라이드 점수의 가중 평균 (Key Point 가중치 / 사실 가중치 / 대본 형태소 수)."""
    def weighted(key: str, weight_key: str):
        pairs = [(e.scores[key], e.scores["_weights"][weight_key]) for e in slide_evaluations if e.scores[key] is not None]
        total = sum(w for _, w in pairs)
        return round(sum(v * w for v, w in pairs) / total, 3) if total else None
    return {
        "content_coverage": weighted("content_coverage", "key_points"),
        "critical_fact_accuracy": weighted("critical_fact_accuracy", "facts"),
        "script_fidelity": weighted("script_fidelity", "script_tokens"),
        "similar_words": sum(e.scores["similar_words"] for e in slide_evaluations),
        "similar_numbers": sum(e.scores["similar_numbers"] for e in slide_evaluations),
    }

## 9. DB 와 파이프라인

대본 분석 노트북과 같은 DB(`outputs/rubrics.sqlite`)에 테이블 세 개를 더합니다.
- `stt_llm_cache`: LLM 응답 캐시 (`kind` = semantic / verifier, 입력 해시, 설정 해시). 같은 입력·프롬프트·모델이면 다시 부르지 않습니다.
- `slide_evaluations`: 연습(take)의 슬라이드별 평가 결과 (`SlideEvaluation` JSON). 어떤 평가 기준(`rubric_id`)으로 채점했는지 함께 남깁니다.
- `similar_items`: **비슷한 말을 한 행씩** — 대본·STT 표현, 대본 문장·위치, STT 문장(T번호)·위치, 원본 STT 위치, 관련 Key Point·Critical Fact,
  규칙 신호·추정. 코칭 agent 는 `load_similar_items(take_id, slide_number)` 로 읽어 판단합니다.

`evaluate_take` 는 연습 한 번을 처리합니다. 모든 슬라이드의 규칙 분석을 먼저 하고, LLM 의미 평가(문장별)를 슬라이드별로 병렬 호출(최대 4개)한 뒤,
충돌한 문장이 있는 슬라이드만 교차 검증을 병렬 호출하고, 마지막에 Key Point 판정을 모아 점수를 냅니다. 평가 결과에는 문장별 판정도 들어 있어
리뷰(코칭) agent 가 "S3 문장을 빠뜨렸다"처럼 구체적으로 짚을 수 있습니다. 한 슬라이드가 실패해도 나머지는 저장되고, 다시 실행하면 캐시에 없는 것만 호출합니다.
`sample=k` 로 부르면 같은 입력을 k 번째로 다시 채점합니다 (12장). 캐시를 따로 쓰고 결과는 저장하지 않습니다.

In [14]:
class SlideEvaluation(BaseModel):
    """Structured Evaluation: 슬라이드 한 장의 평가 결과. 이 모양 그대로 DB 에 저장한다."""
    take_id: str
    script_name: str
    slide_number: int
    rubric_id: str = Field(description="채점에 쓴 평가 기준 (대본 분석 노트북의 rubric_id)")
    scores: dict
    sentences: list[SentenceResult] = Field(description="대본 문장별 판정 (LLM 문장 판정 + 규칙 + 교차 검증)")
    key_points: list[KeyPointResult] = Field(description="Key Point 판정 = 근거 문장 판정을 모은 것")
    critical_facts: list[FactCheck]
    similar_items: list[SimilarItem] = Field(description="비슷한 말 — 비율에서 뺀 판단 보류 항목 (위치 포함)")
    fidelity: dict = Field(description="대본 충실도 세부 (recall / precision / fidelity)")
    alignments: list[Alignment]
    verification: dict = Field(description="검증 필요 여부와 대상 Key Point")
    stt_sentences: list[str] = Field(description="정규화한 STT 문장 (근거 번호 T0, T1 … 의 원문)")
    created_at: str


conn.executescript("""
    -- STT 평가용 LLM 응답 캐시. kind = semantic / verifier (반복 채점의 k 번째 응답은 semantic#k …)
    CREATE TABLE IF NOT EXISTS stt_llm_cache (
        kind        TEXT NOT NULL,
        input_hash  TEXT NOT NULL,
        config_hash TEXT NOT NULL,
        output_json TEXT NOT NULL,
        created_at  TEXT NOT NULL,
        PRIMARY KEY (kind, input_hash, config_hash)
    );
    -- 연습(take) 한 번의 슬라이드별 평가 결과
    CREATE TABLE IF NOT EXISTS slide_evaluations (
        take_id         TEXT    NOT NULL,
        slide_number    INTEGER NOT NULL,
        script_name     TEXT    NOT NULL,
        rubric_id       TEXT    NOT NULL,
        evaluation_json TEXT    NOT NULL,
        created_at      TEXT    NOT NULL,
        PRIMARY KEY (take_id, slide_number)
    );
    -- 비슷한 말: 코칭 agent 가 위치로 찾아 판단할 수 있게 한 행씩 저장한다
    CREATE TABLE IF NOT EXISTS similar_items (
        take_id               TEXT    NOT NULL,
        slide_number          INTEGER NOT NULL,
        item_id               TEXT    NOT NULL,
        kind                  TEXT    NOT NULL,   -- word / number
        script_text           TEXT    NOT NULL,
        stt_text              TEXT    NOT NULL,
        script_sentence_index INTEGER NOT NULL,
        script_start          INTEGER NOT NULL,   -- 정규화 대본 텍스트 기준
        script_end            INTEGER NOT NULL,
        stt_sentence_index    INTEGER NOT NULL,   -- 정규화 STT 문장 번호 (T번호)
        stt_start             INTEGER NOT NULL,   -- 정규화 STT 텍스트 기준
        stt_end               INTEGER NOT NULL,
        stt_raw_start         INTEGER,            -- 원본 STT 텍스트 기준 (Deepgram 단어 타임스탬프와 맞출 때)
        stt_raw_end           INTEGER,
        fact_id               TEXT,
        key_point_ids         TEXT    NOT NULL,   -- JSON 배열
        rule_guess            TEXT    NOT NULL,   -- 규칙 추정 (참고)
        signals               TEXT    NOT NULL,   -- JSON 배열
        created_at            TEXT    NOT NULL,
        PRIMARY KEY (take_id, slide_number, item_id)
    );
""")


def _cached_call(kind: str, message: str, config_hash: str, model_cls):
    row = conn.execute("SELECT output_json FROM stt_llm_cache WHERE kind = ? AND input_hash = ? AND config_hash = ?",
                       (kind, _message_hash(message), config_hash)).fetchone()
    return model_cls.model_validate_json(row[0]) if row else None


def _save_call(kind: str, message: str, config_hash: str, output: BaseModel) -> None:
    conn.execute("INSERT OR REPLACE INTO stt_llm_cache VALUES (?, ?, ?, ?, ?)",
                 (kind, _message_hash(message), config_hash, output.model_dump_json(), datetime.now(timezone.utc).isoformat()))
    conn.commit()


def _llm_step(kind: str, messages: dict, fn, llm_obj, config_hash: str, model_cls) -> tuple[dict, dict, int]:
    """캐시에 없는 메시지만 병렬로 호출하고 저장한다. (결과, 실패, 호출 수)"""
    results = {k: _cached_call(kind, m, config_hash, model_cls) for k, m in messages.items()}
    jobs = {k: (messages[k], llm_obj) for k, r in results.items() if r is None}
    done, errors = _call_in_parallel(jobs, fn, 4)
    for k, output in done.items():
        results[k] = output
        _save_call(kind, messages[k], config_hash, output)
    return results, errors, len(jobs)


def save_evaluation(evaluation: SlideEvaluation) -> None:
    conn.execute("INSERT OR REPLACE INTO slide_evaluations VALUES (?, ?, ?, ?, ?, ?)",
                 (evaluation.take_id, evaluation.slide_number, evaluation.script_name, evaluation.rubric_id,
                  evaluation.model_dump_json(), evaluation.created_at))
    conn.execute("DELETE FROM similar_items WHERE take_id = ? AND slide_number = ?", (evaluation.take_id, evaluation.slide_number))
    conn.executemany(
        "INSERT INTO similar_items VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        [(evaluation.take_id, evaluation.slide_number, it.item_id, it.kind, it.script_text, it.stt_text,
          it.script_sentence_index, *it.script_span, it.stt_sentence_index, *it.stt_span, *(it.stt_raw_span or (None, None)),
          it.fact_id, json.dumps(it.key_point_ids), it.rule_guess,
          json.dumps(it.signals, ensure_ascii=False), evaluation.created_at)
         for it in evaluation.similar_items])
    conn.commit()


def load_similar_items(take_id: str, slide_number: int | None = None) -> pd.DataFrame:
    """코칭 agent 용: 연습 한 번(또는 슬라이드 한 장)의 비슷한 말을 위치와 함께 읽는다."""
    query, params = "SELECT * FROM similar_items WHERE take_id = ?", [take_id]
    if slide_number is not None:
        query, params = query + " AND slide_number = ?", params + [slide_number]
    return pd.read_sql_query(query + " ORDER BY slide_number, stt_start", conn, params=params)


def evaluate_take(take: Take, eval_llm, verifier_llm, sample: int = 0) -> tuple[list[SlideEvaluation], dict]:
    """Presentation Evaluation Pipeline. 발표 연습 한 번(STT)을 대본의 평가 기준으로 채점한다.

    ① STT 정규화 → ② 규칙 분석 (문장 정렬 · 핵심 사실 검증 · 비슷한 말 찾기 · 대본 충실도) ∥ LLM 의미 평가 (대본 문장별, 슬라이드당 1회)
    → ③ 문장별 병합·충돌 검사 (코드) → ④ 충돌한 문장이 있는 슬라이드만 LLM 교차 검증 (슬라이드당 1회)
    → ⑤ Key Point 판정 = 문장 판정 모으기 (코드) → 점수 계산 (코드) → ⑥ 저장 (평가 결과 + 비슷한 말 위치).
    대본을 다시 분석하지 않고 DB 의 평가 기준을 읽어 쓴다.
    sample > 0 이면 같은 입력을 k 번째로 다시 채점한다 (12장 반복 채점). 캐시는 따로 쓰고 결과는 저장하지 않는다.
    """
    suffix = f"#{sample}" if sample else ""
    slides = {}
    for s in take.slides:
        rubric = load_rubric(conn, take.script_name, s.slide_number)
        if rubric is None:
            raise ValueError(f"{take.script_name} 슬라이드 {s.slide_number} 의 평가 기준이 없다 — 대본 분석 노트북을 먼저 실행한다")
        script_norm = normalize_script(rubric.normalized_script)
        stt_norm = normalize_stt(s.stt)
        script_ptokens, stt_ptokens = positioned_tokens(script_norm), positioned_tokens(stt_norm)
        alignments = align_sentences(script_norm, [(t.key, t.sentence) for t in script_ptokens], stt_norm,
                                     [(t.key, t.sentence) for t in stt_ptokens], rubric.sentence_roles)
        fact_checks = check_critical_facts(rubric, stt_norm, alignments)
        annotate_mismatches(fact_checks, rubric, stt_norm)
        items = find_similar_items(rubric, script_norm, script_ptokens, stt_norm, stt_ptokens, alignments, fact_checks, s.stt)
        settle_facts(fact_checks, items)
        # 대본 충실도: 비슷한 말 자리의 형태소는 양쪽에서 뺀다
        script_tokens = without_spans(script_ptokens, [it.script_span for it in items])
        fidelity = script_fidelity(script_tokens, without_spans(stt_ptokens, [it.stt_span for it in items]), rubric.sentence_roles)
        fidelity["script_tokens"] = sum(1 for _, i in script_tokens if rubric.sentence_roles[i] != "skip")
        unverified = [c for c in fact_checks if c.status == "unverified"]
        slides[s.slide_number] = dict(rubric=rubric, script_norm=script_norm, stt_norm=stt_norm, alignments=alignments,
                                      fidelity=fidelity, fact_checks=fact_checks, unverified=unverified, items=items)

    # ② LLM 의미 평가 (규칙 분석과 독립: 규칙 결과를 입력에 넣지 않는다)
    messages = {n: semantic_eval_message(n, d["rubric"], d["stt_norm"], d["unverified"]) for n, d in slides.items()}
    semantics, sem_errors, sem_calls = _llm_step("semantic" + suffix, messages, evaluate_semantics, eval_llm, eval_config_hash(), SemanticEvaluation)

    # ③ 문장별 병합·충돌 검사
    for n, d in slides.items():
        if n in sem_errors:
            continue
        resolve_name_checks(d["fact_checks"], d["unverified"], semantics[n])
        d["sentences"] = merge_sentences(d["rubric"], semantics[n], d["fact_checks"], d["alignments"], d["stt_norm"], d["items"])

    # ④ 교차 검증: 충돌한 문장이 있는 슬라이드만, 슬라이드마다 한 번
    ver_messages = {
        n: verifier_message(d["rubric"], d["sentences"], d["fact_checks"], d["alignments"], d["stt_norm"], d["items"])
        for n, d in slides.items() if "sentences" in d and any(s.conflicts for s in d["sentences"])
    }
    verified, ver_errors, ver_calls = _llm_step("verifier" + suffix, ver_messages, verify, verifier_llm, verifier_config_hash(), VerifierResult)

    # ⑤ Key Point 판정(문장 판정 모으기) → 점수 → ⑥ 저장
    evaluations = []
    for n, d in slides.items():
        if "sentences" not in d or n in ver_errors:
            continue
        apply_verification(d["sentences"], verified.get(n))
        results = aggregate_key_points(d["rubric"], d["sentences"], d["fact_checks"], d["items"], d["stt_norm"])
        evaluation = SlideEvaluation(
            take_id=take.take_id, script_name=take.script_name, slide_number=n, rubric_id=d["rubric"].meta.rubric_id,
            scores=slide_scores(results, d["fact_checks"], d["fidelity"], d["items"]),
            sentences=d["sentences"], key_points=results, critical_facts=d["fact_checks"], similar_items=d["items"],
            fidelity=d["fidelity"], alignments=d["alignments"],
            verification={"required": n in ver_messages, "sentences": [s.sentence_index for s in d["sentences"] if s.conflicts]},
            stt_sentences=[s.text for s in d["stt_norm"].sentences],
            created_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
        )
        if not sample:
            save_evaluation(evaluation)
        evaluations.append(evaluation)

    failed = {n: f"의미 평가 {e}" for n, e in sem_errors.items()} | {n: f"교차 검증 {e}" for n, e in ver_errors.items()}
    stats = {"slides": len(slides), "semantic_calls": sem_calls, "verifier_calls": ver_calls,
             "verified_slides": len(ver_messages), "failed": failed}
    return evaluations, stats

## 10. 실행

연습 18번(대본 2개 × 9번)을 채점합니다. `semantic_calls` / `verifier_calls` 는 이번 실행에서 실제로 API 를 부른 횟수(캐시 제외)입니다.

In [15]:
eval_llm = build_eval_llm()                  # 의미 평가
verifier_llm = build_verifier_llm()          # 교차 검증

all_evaluations: dict[str, list[SlideEvaluation]] = {}
failures = {}
for take in takes:
    evaluations, stats = evaluate_take(take, eval_llm, verifier_llm)
    all_evaluations[take.take_id] = evaluations
    failures[take.take_id] = stats.pop("failed")
    print(take.take_id, stats, "실패한 슬라이드:", sorted(failures[take.take_id]) or "-")

first_error = next((msg for failed in failures.values() for msg in failed.values()), None)
if first_error:
    raise RuntimeError(f"LLM 호출 실패 — 첫 오류: {first_error}")


def take_summary() -> pd.DataFrame:
    rows = []
    for take in takes:
        evaluations = all_evaluations[take.take_id]
        statuses = Counter(r.status for e in evaluations for r in e.key_points)
        rows.append({
            "take_id": take.take_id, "scenario": take.scenario, **take_scores(evaluations),
            "covered/partial/missing/contradicted": "/".join(str(statuses[s]) for s in ("covered", "partial", "missing", "contradicted")),
            "검증한 슬라이드": sum(e.verification["required"] for e in evaluations),
            "틀리게 말한 수치": sum(c.status == "mismatched" for e in evaluations for c in e.critical_facts),
        })
    return pd.DataFrame(rows)


take_summary()

가상대본1_take1 {'slides': 9, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 0} 실패한 슬라이드: -


가상대본1_take2 {'slides': 9, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 0} 실패한 슬라이드: -


가상대본1_take3 {'slides': 9, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 0} 실패한 슬라이드: -


가상대본1_take4 {'slides': 9, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 6} 실패한 슬라이드: -


가상대본1_take5 {'slides': 9, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 3} 실패한 슬라이드: -


...

가상대본2_take6 {'slides': 11, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 6} 실패한 슬라이드: -


가상대본2_take7 {'slides': 11, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 1} 실패한 슬라이드: -


가상대본2_take8 {'slides': 11, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 1} 실패한 슬라이드: -


가상대본2_take9 {'slides': 11, 'semantic_calls': 0, 'verifier_calls': 0, 'verified_slides': 4} 실패한 슬라이드: -


,take_id,scenario,content_coverage,critical_fact_accuracy,script_fidelity,similar_words,similar_numbers,covered/partial/missing/contradicted,검증한 슬라이드,틀리게 말한 수치
0,가상대본1_take1,충실,1.000,1.000,0.918,0,0,39/0/0/0,0,0
1,가상대본1_take2,의역,0.956,0.961,0.537,0,0,34/5/0/0,0,0
2,가상대본1_take3,누락,0.563,0.505,0.683,0,0,14/14/11/0,0,0
3,가상대본1_take4,실수,0.791,0.922,0.889,0,0,33/0/0/6,6,4
4,가상대본1_take5,혼합,0.778,0.802,0.647,0,1,29/3/5/2,3,1
5,가상대본1_take6,인식오류,0.905,0.947,0.903,0,5,37/0/0/2,6,2
6,가상대본1_take7,더듬기,0.937,0.981,0.890,0,0,35/1/3/0,0,0
7,가상대본1_take8,요약,0.570,0.558,0.438,0,0,13/13/13/0,0,0
8,가상대본1_take9,발음,0.924,1.000,0.965,6,2,37/0/0/2,4,0
9,가상대본2_take1,충실,1.000,1.000,0.904,0,0,41/0/0/0,1,0


### 비슷한 말 목록 (코칭 agent 용)

비율 점수에서 뺀 비슷한 말을 DB(`similar_items`)에서 읽은 목록입니다. 코칭 agent 는 위치로 대본 문장, STT 문장(T번호),
원본 STT 구간(→ Deepgram 단어 타임스탬프 → 녹음 구간)을 찾아, 발표자가 실제로 잘못 말했는지 판단합니다.
`규칙 추정` 과 `규칙 신호` 는 판단을 돕는 참고 정보입니다.

In [16]:
def similar_report(take_id: str) -> pd.DataFrame:
    """비슷한 말 목록 (DB `similar_items` 에서 읽음). 코칭 agent 는 이 위치로 대본·STT·녹음 구간을 찾아 판단한다."""
    df = load_similar_items(take_id)
    raw = {s.slide_number: s.stt for s in next(t for t in takes if t.take_id == take_id).slides}
    return pd.DataFrame([{
        "slide": r.slide_number, "종류": "수치" if r.kind == "number" else "단어", "대본": r.script_text, "STT": r.stt_text,
        "대본 위치": f"S{r.script_sentence_index} [{r.script_start}:{r.script_end}]",
        "STT 위치": f"T{r.stt_sentence_index} [{r.stt_start}:{r.stt_end}]",
        "원본 STT 위치": (f"[{int(r.stt_raw_start)}:{int(r.stt_raw_end)}] '{raw[r.slide_number][int(r.stt_raw_start):int(r.stt_raw_end)]}'"
                        if pd.notna(r.stt_raw_start) else "-"),
        "Key Point": ", ".join(json.loads(r.key_point_ids)), "규칙 추정": r.rule_guess, "규칙 신호": " / ".join(json.loads(r.signals)[-1:]),
    } for r in df.itertuples()])


# 연습마다 비슷한 말 개수 (비율에서 뺀 판단 보류 항목)
display(pd.read_sql_query(
    "SELECT take_id, SUM(kind = 'word') AS 비슷한_단어, SUM(kind = 'number') AS 비슷한_수치 FROM similar_items GROUP BY take_id", conn))
# 예: 음성 인식이 발음이 비슷한 단어를 잘못 적은 연습
display(similar_report("가상대본1_take9"))

,take_id,비슷한_단어,비슷한_수치
0,가상대본1_take5,0,1
1,가상대본1_take6,0,5
2,가상대본1_take9,6,2
3,가상대본2_take4,0,1
4,가상대본2_take5,0,1
5,가상대본2_take6,0,5
6,가상대본2_take9,7,1


,slide,종류,대본,STT,대본 위치,STT 위치,원본 STT 위치,Key Point,규칙 추정,규칙 신호
0,2,단어,좌석,자석,S0 [9:11],T0 [11:13],[11:13] '자석',KP1,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
1,2,수치,약 18퍼센트,약 팔 퍼센트,S2 [125:132],T2 [121:128],[121:128] '약 팔 퍼센트',KP3,asr_error,음절 하나('십')가 빠짐 → 음성 인식 오류일 수 있음
2,3,수치,30분,삼 분,S2 [115:118],T2 [110:113],[110:113] '삼 분',KP3,asr_error,음절 하나('십')가 빠짐 → 음성 인식 오류일 수 있음
3,3,단어,센서,센터,S4 [219:221],T4 [212:214],[214:216] '센터',KP5,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
4,5,단어,이용률,이용료,S4 [171:174],T4 [170:173],[170:173] '이용료',"KP5, KP6",asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
5,6,단어,노쇼,노소,S3 [143:145],T3 [148:150],[148:150] '노소',KP4,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
6,8,단어,리포트,리포터,S2 [93:96],T2 [87:90],[87:90] '리포터',KP3,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴
7,9,단어,대전,대천,S1 [33:35],T1 [36:38],[36:38] '대천',KP1,asr_error,대본 단어가 이 자리에서 빠지고 발음이 비슷한 다른 말이 나옴


### 슬라이드 상세

대본 문장별 1차 판정 · 최종 판정 · 근거 · 단어 비율 · 충돌 이유, 그 판정을 모은 Key Point 판정, 사실 검증 결과, 비슷한 말입니다.

In [17]:
def show_evaluation(take_id: str, slide_number: int) -> None:
    e = next(x for x in all_evaluations[take_id] if x.slide_number == slide_number)
    print(f"[{take_id} / 슬라이드 {slide_number}] 점수:", {k: v for k, v in e.scores.items() if not k.startswith("_")})
    for i, text in enumerate(e.stt_sentences):
        print(f"  [T{i}] {text}")
    print("대본 문장별 판정")
    display(pd.DataFrame([
        {"문장": f"S{s.sentence_index}", "역할": s.role, "1차": s.first_status, "최종": s.status, "검증": s.verified,
         "근거": s.evidence, "단어 비율": s.lexical_coverage, "충돌": ", ".join(s.conflicts), "이유": s.reason}
        for s in e.sentences
    ]))
    print("Key Point 판정 (근거 문장 판정을 모은 것)")
    display(pd.DataFrame([
        {"id": r.key_point_id, "importance": r.importance, "근거 문장": [f"S{i}" for i in r.sentence_indices],
         "1차": r.first_status, "최종": r.status, "검증": r.verified, "내용": r.content}
        for r in e.key_points
    ]))
    display(pd.DataFrame([
        {"id": c.fact_id, "type": c.type, "대본": c.value, "결과": c.status, "STT": c.stt_value,
         "원인": c.cause or "", "note": c.cause_reason or c.note}
        for c in e.critical_facts
    ]))
    if e.similar_items:
        print("비슷한 말 (비율에서 뺌 — 코칭 agent 가 판단)")
        display(pd.DataFrame([
            {"id": it.item_id, "종류": it.kind, "대본": it.script_text, "STT": it.stt_text,
             "대본 위치": f"S{it.script_sentence_index} {list(it.script_span)}", "STT 위치": f"T{it.stt_sentence_index} {list(it.stt_span)}",
             "Key Point": ", ".join(it.key_point_ids), "규칙 추정": it.rule_guess}
            for it in e.similar_items
        ]))


# 잘못 말한 수치와 말을 고쳐 말한 경우가 섞인 슬라이드
show_evaluation("가상대본1_take4", 6)
# 음성 인식이 수치 두 개를 발음이 비슷한 다른 값으로 적은 슬라이드 (천이백 → 천백, 이십사억 → 십사억) — 비율에서 빠진다
show_evaluation("가상대본1_take6", 7)
# 핵심 주장 문장을 빠뜨린 슬라이드
show_evaluation("가상대본1_take3", 7)

[가상대본1_take4 / 슬라이드 6] 점수: {'content_coverage': 0.7, 'critical_fact_accuracy': 0.92, 'script_fidelity': 0.963, 'similar_words': 0, 'similar_numbers': 0}
  [T0] 이천이십오 년 삼 월부터 8주 동안 제휴 도서관 3곳에서 시범 운영을 했습니다
  [T1] 참여한 이용자는 1860명이었고 빈자리를 찾는 데 걸린 시간은 평균 23분에서 십삼 분으로 약 이십사 퍼센트 아니 사십이 퍼센트 줄었습니다.
  [T2] 알림을 받고 10분 안에 자리를 잡은 비율은 67퍼센트였습니다
  [T3] 노쇼 비율도 18퍼센트에서 칠 퍼센트로 낮아졌고요 그 시범 운영 뒤에 설문에서 이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답했습니다
대본 문장별 판정


,문장,역할,1차,최종,검증,근거,단어 비율,충돌,이유
0,S0,detail,said,said,False,[0],1.000,,2025년 3월부터 8주 동안 제휴 도서관 3곳에서 시범 운영했다는 내용과 핵심 수치가 모두 전달되었습니다.
1,S1,claim,said,said,False,[1],1.000,,"참여자 1,860명과 평균 시간이 23분에서 13분으로 줄었다는 내용, 약 42퍼센트라는 핵심 수치가 전달되었습니다."
2,S2,evidence,contradicted,contradicted,True,[2],0.833,contradicted,"[검증] STT에서 ‘10분 안에 자리를 잡은 비율’이라는 내용은 전달했지만, 대본의 핵심 수치 76%가 아니라 67%라고 말해 수치가 ..."
3,S3,evidence,said,said,False,[3],1.000,,노쇼 비율이 18%에서 7%로 낮아졌다는 내용과 핵심 수치가 모두 전달되었습니다.
4,S4,evidence,said,said,False,[3],1.000,,시범 운영 뒤 설문에서 이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답했다는 내용이 전달되었습니다.


Key Point 판정 (근거 문장 판정을 모은 것)


,id,importance,근거 문장,1차,최종,검증,내용
0,KP1,normal,[S0],covered,covered,False,2025년 3월부터 8주 동안 제휴 도서관 3곳에서 시범 운영을 했다.
1,KP2,critical,[S1],covered,covered,False,빈자리를 찾는 데 걸린 시간이 평균 23분에서 13분으로 약 42퍼센트 줄었다.
2,KP3,high,[S2],contradicted,contradicted,True,알림을 받고 10분 안에 자리를 잡은 비율이 76%이다.
3,KP4,high,[S3],covered,covered,False,노쇼 비율이 18%에서 7%로 낮아졌다.
4,KP5,high,[S4],covered,covered,False,시범 운영 뒤 설문에서 이용자의 85퍼센트 이상이 계속 쓰고 싶다고 답했다.


,id,type,대본,결과,STT,원인,note
0,CF1,date,2025년 3월,matched,이천이십오 년 삼 월,,
1,CF2,duration,8주 동안,matched,8주 동안,,
2,CF3,quantity,3곳,matched,3곳,,
3,CF4,quantity,"1,860명",matched,1860명,,
4,CF5,duration,평균 23분,matched,평균 23분,,
5,CF6,duration,13분,matched,십삼 분,,
6,CF7,percentage,약 42퍼센트,matched,사십이 퍼센트,,한정어 차이
7,CF8,duration,10분,matched,10분,,
8,CF9,percentage,76%,mismatched,67퍼센트,speaker_error,발음이 비슷하지 않거나 음절 순서만 바뀜 — 발표자가 다르게 말함
9,CF10,percentage,18%,matched,18퍼센트,,


...

[가상대본1_take3 / 슬라이드 7] 점수: {'content_coverage': 0.333, 'critical_fact_accuracy': 0.267, 'script_fidelity': 0.6, 'similar_words': 0, 'similar_numbers': 0}
  [T0] 국내 공공도서관은 약 천이백 곳이고요
  [T1] 대학 도서관도 상당히 많습니다
  [T2] 그러니까 처음에는 이용자가 가장 많은 서울이랑 대전의 대학 도서관부터 시작하겠습니다
대본 문장별 판정


,문장,역할,1차,최종,검증,근거,단어 비율,충돌,이유
0,S0,evidence,partial,partial,False,"[0, 1]",0.857,,"공공도서관 약 1,200곳은 말했지만 대학 도서관 약 430곳이라는 핵심 수치는 말하지 않았습니다."
1,S1,claim,missing,missing,False,[],0.000,,초기 대상 약 900곳과 연간 시장 규모 약 24억 원에서 36억 원이라는 내용이 STT에 없습니다.
2,S2,detail,said,said,False,[2],1.000,,이용자가 가장 많은 서울과 대전의 대학 도서관부터 시작하겠다는 뜻을 그대로 전달했습니다.


Key Point 판정 (근거 문장 판정을 모은 것)


,id,importance,근거 문장,1차,최종,검증,내용
0,KP1,high,[S0],partial,partial,False,"국내 공공도서관은 약 1,200곳이고 대학 도서관은 약 430곳이다."
1,KP2,critical,[S1],missing,missing,False,열람실 좌석 예약 시스템을 이미 쓰는 초기 대상은 약 900곳이며 연간 시장 규모는 약 24억 원에서 36억 원으로 추산했다.
2,KP3,normal,[S2],covered,covered,False,초기에는 이용자가 가장 많은 서울과 대전의 대학 도서관부터 시작한다.


,id,type,대본,결과,STT,원인,note
0,CF1,quantity,"약 1,200곳",matched,약 천이백 곳,,
1,CF2,quantity,약 430곳,missing,NaN,,
2,CF3,quantity,약 900곳,missing,NaN,,
3,CF4,money,약 24억 원,missing,NaN,,
4,CF5,money,36억 원,missing,NaN,,
5,CF6,proper_noun,서울,matched,서울,,
6,CF7,proper_noun,대전,matched,대전,,


## 11. 정답 라벨과 비교 (성능 측정)

정답 라벨(대본 문장 단위)로 문장·Key Point·사실의 정답을 만들고, 판정을 비교합니다.
- **문장 정답**: 라벨 그대로 (verbatim·paraphrased → said, partial, missing, contradicted). 라벨과 같은 단위라 가장 직접적인 비교입니다.
- Key Point 정답: 근거 문장 중 하나라도 contradicted 면 contradicted, 모두 전달(verbatim / paraphrased)이면 covered, 모두 missing 이면 missing, 나머지는 partial.
  라벨은 발표자가 실제로 말한 것 기준이라, 음성 인식이 잘못 적은 문장도 전달입니다.
- 사실 정답: 한 번이라도 맞게 말했으면 matched, 음성 인식이 잘못 적었으면 asr, 어림해 말했으면 approximate, 틀린 값을 말했으면 mismatched, 아니면 missing.

| 방법 | 내용 |
|---|---|
| 규칙만 | LLM 없이 규칙 결과로만 문장 판정: 발표자가 다르게 말한 수치 → contradicted, 단어 비율 0.7 이상이고 빠지거나 어림한 사실이 없으면 said, 0.3 이상 partial, 나머지 missing (Key Point 는 이를 모음) |
| LLM 1차 | 5장 문장 판정 그대로 (Key Point 는 이를 모음) |
| 최종 | 충돌 검사(6장)와 교차 검증(7장)을 거친 판정 |

Key Point 정답을 만드는 규칙과 6장의 모으는 규칙이 같습니다. 그래서 Key Point 정확도에는 이 정렬 효과가 섞여 있고, **문장 정확도**가 판정 자체의 성능입니다.

- **사실 정확도**는 판단 보류(`held` = `sound_alike`)를 뺀 사실에서 잽니다. 인식 오류로 적힌 영문 이름을 5장 이름 확인이 '말함'으로 받아들인 경우도 맞음입니다.
- **비슷한 말**: 라벨의 인식 오류(단어·수치, 영문 이름 제외)를 보류했는지(보류율), 보류한 것 중 실제 인식 오류와 실제 발표자 실수가 얼마나 되는지
  (발표자 실수는 코칭 agent 가 가려야 할 몫), 규칙 추정이 실제 원인을 맞힌 비율을 봅니다.

'심각한 오판'은 전달(covered)과 누락·모순을 뒤바꾼 경우입니다. LLM 판정은 응답에 따라 달라지므로 아래 수치는 이 실행의 결과입니다.

**라벨의 한계** — 정답 라벨은 대본 **문장** 단위라서, Key Point 정답은 문장 라벨로부터 계산한 근사값입니다.
한 문장에 Key Point 두 개가 들어 있고 그중 하나만 말한 경우나, Key Point 근거에 전환 문장이 섞여 그 문장만 빠진 경우에는 판정이 맞아도 오답으로 셉니다.
**발음이 비슷한 발표자 실수**(`18%` → `28%`)는 이 평가에서 보류하므로, Key Point 정답(contradicted)과 다르게 나옵니다 — 코칭 agent 가 가릴 몫입니다.

In [18]:
# 정답 라벨: 가상 STT 를 만들 때 대본 문장마다 어떻게 말했는지 기록해 둔 것 (평가 파이프라인은 보지 않는다)
LABEL_STATUS = {"verbatim": "covered", "paraphrased": "covered", "partial": "partial", "missing": "missing", "contradicted": "contradicted"}
LABEL_SENTENCE = {"verbatim": "said", "paraphrased": "said", "partial": "partial", "missing": "missing", "contradicted": "contradicted"}
ORDER = ["covered", "partial", "missing", "contradicted"]
SENTENCE_ORDER = ["said", "partial", "missing", "contradicted"]
FACT_TRUTH_ORDER = ["matched", "approximate", "asr", "mismatched", "missing"]
FACT_PRED_ORDER = ["matched", "approximate", "held", "mismatched", "missing"]
# 예측 이름: sound_alike(발음이 비슷한 말 — 판단 보류) → held
FACT_PRED = {"matched": "matched", "approximate": "approximate", "sound_alike": "held", "mismatched": "mismatched",
             "missing": "missing", "unverified": "missing"}


def load_labels(take_id: str) -> dict[int, dict[int, dict]]:
    data = json.loads((LABEL_DIR / f"{take_id}.json").read_text(encoding="utf-8"))
    return {s["slide_number"]: {x["index"]: x for x in s["sentences"]} for s in data["slides"]}


def truth_key_point(sentence_indices: list[int], labels: dict[int, dict]) -> str:
    """문장 라벨 → Key Point 정답. 모순이 하나라도 있으면 contradicted, 모두 전달이면 covered, 모두 빠졌으면 missing, 나머지 partial.
    라벨은 발표자가 '실제로 말한 것' 기준이라, 음성 인식이 잘못 적은 문장도 전달(verbatim)이다."""
    statuses = [LABEL_STATUS[labels[i]["status"]] for i in sentence_indices if i in labels]
    if not statuses:
        return "missing"
    if "contradicted" in statuses:
        return "contradicted"
    if all(s == "covered" for s in statuses):
        return "covered"
    if all(s == "missing" for s in statuses):
        return "missing"
    return "partial"


def _same_value(fact: CriticalFact, surface: str) -> bool:
    if fact.type in NAME_TYPES:
        return _compact(fact.value)[0] in _compact(surface)[0] or _compact(surface)[0] in _compact(fact.value)[0]
    return any((f.type, f.normalized) == (fact.type, fact.normalized) for f in extract_critical_facts(normalize_script(surface)))


def truth_fact(fact: CriticalFact, labels: dict[int, dict]) -> str:
    """문장 라벨 → 사실 정답: matched / approximate(어림해 말함) / asr(맞게 말했지만 인식 오류) / mismatched(틀리게 말함) / missing."""
    outcomes = []
    for i in set(fact.sentence_indices):
        label = labels.get(i)
        if label is None:
            continue
        if any(_same_value(fact, c["script"]) for c in label.get("asr_errors", [])):
            outcomes.append("asr")
        elif any(_same_value(fact, c["script"]) for c in label.get("approximated_values", [])):
            outcomes.append("approximate")
        elif any(_same_value(fact, c["script"]) for c in label["changed_values"]):
            outcomes.append("mismatched")
        elif label["status"] == "missing" or any(_same_value(fact, v) for v in label["dropped_values"]):
            outcomes.append("missing")
        else:
            outcomes.append("matched")
    return next((o for o in ("matched", "asr", "approximate", "mismatched") if o in outcomes), "missing")


def rule_only_sentence(s: SentenceResult) -> str:
    """비교용 기준선: LLM 없이 규칙 결과(문장의 단어 비율 + 사실 검증)만으로 문장 판정. 판단 보류(sound_alike) 수치는 보지 않는다."""
    statuses = set(s.fact_statuses.values())
    if "mismatched" in statuses:
        return "contradicted"
    if s.lexical_coverage >= 0.7 and not statuses & {"missing", "approximate"}:
        return "said"
    return "partial" if s.lexical_coverage >= 0.3 else "missing"


def coverage_from(statuses_and_importance: list[tuple[str, str]]) -> float:
    weight = sum(SCORE_WEIGHT[imp] for _, imp in statuses_and_importance)
    score = sum(STATUS_SCORE[s] * SCORE_WEIGHT[imp] for s, imp in statuses_and_importance)
    return max(score / weight, 0.0) if weight else 0.0


def build_tables(evaluations_by_take: dict[str, list[SlideEvaluation]]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """채점 결과 한 벌 → (Key Point 표, 사실 표, 슬라이드 표, 문장 표). 각 행에 정답과 방법별 예측을 둔다."""
    kp_rows, fact_rows, slide_rows, sentence_rows = [], [], [], []
    for take in takes:
        labels = load_labels(take.take_id)
        for e in evaluations_by_take.get(take.take_id, []):
            rubric = load_rubric(conn, take.script_name, e.slide_number)
            kps = {kp.id: kp for kp in rubric.key_points}
            facts = {f.id: f for f in rubric.critical_facts}
            rule_only = {s.sentence_index: rule_only_sentence(s) for s in e.sentences}
            for s in e.sentences:
                label = labels[e.slide_number].get(s.sentence_index)
                if label is not None:
                    sentence_rows.append({"take_id": take.take_id, "scenario": take.scenario, "slide": e.slide_number,
                                          "sentence": s.sentence_index, "정답": LABEL_SENTENCE[label["status"]],
                                          "규칙만": rule_only[s.sentence_index], "LLM 1차": s.first_status, "최종": s.status, "검증": s.verified})
            truths = {}
            for r in e.key_points:
                truth = truth_key_point(kps[r.key_point_id].sentence_indices, labels[e.slide_number])
                truths[r.key_point_id] = truth
                kp_rows.append({"take_id": take.take_id, "scenario": take.scenario, "slide": e.slide_number, "kp": r.key_point_id,
                                "정답": truth, "규칙만": aggregate_status([rule_only[i] for i in r.sentence_indices if i in rule_only]),
                                "LLM 1차": r.first_status, "최종": r.status, "검증": r.verified})
            for c in e.critical_facts:
                fact_rows.append({"take_id": take.take_id, "scenario": take.scenario, "slide": e.slide_number, "fact_id": c.fact_id,
                                  "fact": c.value, "type": c.type, "STT": c.stt_value,
                                  "정답": truth_fact(facts[c.fact_id], labels[e.slide_number]), "최종": FACT_PRED[c.status]})
            slide_rows.append({
                "take_id": take.take_id, "scenario": take.scenario, "slide": e.slide_number,
                "정답 coverage": coverage_from([(truths[r.key_point_id], r.importance) for r in e.key_points]),
                "LLM 1차 coverage": coverage_from([(r.first_status, r.importance) for r in e.key_points]),
                "최종 coverage": e.scores["content_coverage"],
            })
    return pd.DataFrame(kp_rows), pd.DataFrame(fact_rows), pd.DataFrame(slide_rows), pd.DataFrame(sentence_rows)


SEVERE = {("covered", "missing"), ("covered", "contradicted"), ("missing", "covered"), ("contradicted", "covered"),
          ("said", "missing"), ("said", "contradicted"), ("missing", "said"), ("contradicted", "said")}


def kp_accuracy(kp_table: pd.DataFrame, column: str) -> dict:
    exact = (kp_table[column] == kp_table["정답"]).mean()
    severe = kp_table.apply(lambda row: (row[column], row["정답"]) in SEVERE, axis=1).mean()
    return {"정확도": round(float(exact), 3), "심각한 오판 비율": round(float(severe), 3)}


def fact_accuracy(fact_table: pd.DataFrame) -> float:
    """사실 정확도 (판단 보류 held 는 빼고). 인식 오류(asr)인 이름을 5장 이름 확인이 '말함(matched)'으로 받아들인 경우도 맞음 (감점 없음)."""
    rows = fact_table[fact_table["최종"] != "held"]
    ok = (rows["정답"] == rows["최종"]) | ((rows["정답"] == "asr") & (rows["최종"] == "matched"))
    return round(float(ok.mean()), 3)


# ── 비슷한 말 (4-3) 평가 ─────────────────────────────────────
LABEL_CAUSE = {"asr_errors": "asr_error", "changed_values": "speaker_error", "approximated_values": "approximation"}


def label_pairs(take_id: str) -> dict[int, list[tuple[str, str, str]]]:
    """슬라이드 → [(원인, 대본 표현, STT 표현)]. 라벨의 인식 오류 / 발표자가 바꿔 말함 / 어림."""
    data = json.loads((LABEL_DIR / f"{take_id}.json").read_text(encoding="utf-8"))
    return {s["slide_number"]: [(LABEL_CAUSE[key], c["script"], c["stt"]) for x in s["sentences"] for key in LABEL_CAUSE for c in x.get(key, [])]
            for s in data["slides"]}


def _pair_match(item: SimilarItem, script: str, stt: str) -> bool:
    a, b, c, d = _plain(item.script_text), _plain(script), _plain(item.stt_text), _plain(stt)
    return bool(a and c) and (a in b or b in a) and (c in d or d in c)


def similar_tables(evaluations_by_take: dict[str, list[SlideEvaluation]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(항목 표, 라벨 표). 항목 표: 비슷한 말마다 실제 원인(라벨에 없으면 none) / 규칙 추정.
    라벨 표: 라벨의 인식 오류마다 비슷한 말로 보류했는지. 영문 이름은 5장 이름 확인이 따로 봐서 뺀다."""
    item_rows, label_rows = [], []
    for take in takes:
        pairs = label_pairs(take.take_id)
        for e in evaluations_by_take.get(take.take_id, []):
            slide_pairs = pairs.get(e.slide_number, [])
            for it in e.similar_items:
                truth = next((cause for cause, script, stt in slide_pairs if _pair_match(it, script, stt)), "none")
                item_rows.append({"take_id": take.take_id, "slide": e.slide_number, "종류": it.kind, "대본": it.script_text,
                                  "STT": it.stt_text, "실제": truth, "규칙 추정": it.rule_guess})
            for cause, script, stt in slide_pairs:
                if cause != "asr_error" or re.search(r"[A-Za-z]", script):
                    continue
                hit = any(_pair_match(it, script, stt) for it in e.similar_items)
                label_rows.append({"take_id": take.take_id, "slide": e.slide_number, "대본": script, "STT": stt, "보류함": hit})
    return pd.DataFrame(item_rows), pd.DataFrame(label_rows)


def similar_accuracy(evaluations_by_take: dict[str, list[SlideEvaluation]]) -> dict:
    items, labels = similar_tables(evaluations_by_take)
    return {"인식 오류 보류율": round(float(labels["보류함"].mean()), 3) if len(labels) else float("nan"),
            "보류 중 실제 인식 오류": round(float((items["실제"] == "asr_error").mean()), 3) if len(items) else float("nan")}


kp_table, fact_table, slide_table, sentence_table = build_tables(all_evaluations)

print("대본 문장 판정 (전체", len(sentence_table), "개) — 라벨과 같은 단위라 가장 직접적인 비교")
display(pd.DataFrame({col: kp_accuracy(sentence_table, col) for col in ("규칙만", "LLM 1차", "최종")}).T)
print("시나리오별 최종 정확도:", sentence_table.groupby("scenario").apply(lambda g: round(float((g["최종"] == g["정답"]).mean()), 3)).to_dict())
display(pd.crosstab(sentence_table["정답"], sentence_table["최종"]).reindex(index=SENTENCE_ORDER, columns=SENTENCE_ORDER, fill_value=0))

print("Key Point 판정 (전체", len(kp_table), "개)")
display(pd.DataFrame({col: kp_accuracy(kp_table, col) for col in ("규칙만", "LLM 1차", "최종")}).T)
print("시나리오별 최종 정확도:", kp_table.groupby("scenario").apply(lambda g: round(float((g["최종"] == g["정답"]).mean()), 3)).to_dict())
print("최종 판정 혼동 행렬 (행 = 정답, 열 = 예측)")
display(pd.crosstab(kp_table["정답"], kp_table["최종"]).reindex(index=ORDER, columns=ORDER, fill_value=0))

held = fact_table[fact_table["최종"] == "held"]
print(f"핵심 사실 검증 (전체 {len(fact_table)}개, 판단 보류 {len(held)}개 제외) 정확도:", fact_accuracy(fact_table))
display(pd.crosstab(fact_table["정답"], fact_table["최종"]).reindex(index=FACT_TRUTH_ORDER, columns=FACT_PRED_ORDER, fill_value=0))

sim_items, sim_labels = similar_tables(all_evaluations)
print(f"비슷한 말 — 라벨의 인식 오류 {len(sim_labels)}개 (영문 이름 제외), 보류한 비슷한 말 {len(sim_items)}개")
display(pd.DataFrame([{
    "인식 오류를 보류한 비율": similar_accuracy(all_evaluations)["인식 오류 보류율"],
    "보류한 것 중 실제 인식 오류": similar_accuracy(all_evaluations)["보류 중 실제 인식 오류"],
    "보류한 것 중 실제 발표자 실수 (코칭 agent 가 가려야 할 것)": round(float((sim_items["실제"] == "speaker_error").mean()), 3),
    "규칙 추정 정확도": round(float((sim_items["실제"] == sim_items["규칙 추정"]).mean()), 3),
}]).T.rename(columns={0: "값"}))
print("보류한 비슷한 말의 실제 원인 (행) × 규칙 추정 (열)")
display(pd.crosstab(sim_items["실제"], sim_items["규칙 추정"]))
print("보류하지 못한 인식 오류")
display(sim_labels[~sim_labels["보류함"]])

slide_table["LLM 1차 오차"] = (slide_table["LLM 1차 coverage"] - slide_table["정답 coverage"]).abs()
slide_table["최종 오차"] = (slide_table["최종 coverage"] - slide_table["정답 coverage"]).abs()
print("슬라이드 Content Coverage 평균 절대 오차 (0~1):",
      {"LLM 1차": round(float(slide_table["LLM 1차 오차"].mean()), 3), "최종": round(float(slide_table["최종 오차"].mean()), 3)})
display(slide_table.groupby("scenario")[["정답 coverage", "최종 coverage", "최종 오차"]].mean().round(3))

print("틀린 판정 목록 (최종 기준)")
display(kp_table[kp_table["최종"] != kp_table["정답"]][["take_id", "slide", "kp", "정답", "LLM 1차", "최종", "검증"]])
display(fact_table[(fact_table["최종"] != fact_table["정답"]) & (fact_table["최종"] != "held")
                   & ~((fact_table["정답"] == "asr") & (fact_table["최종"] == "matched"))][["take_id", "slide", "fact", "STT", "정답", "최종"]])

대본 문장 판정 (전체 765 개) — 라벨과 같은 단위라 가장 직접적인 비교


,정확도,심각한 오판 비율
규칙만,0.850,0.020
LLM 1차,0.967,0.024
최종,0.986,0.005


시나리오별 최종 정확도: {'누락': 0.953, '더듬기': 1.0, '발음': 0.988, '실수': 0.988, '요약': 0.965, '의역': 1.0, '인식오류': 1.0, '충실': 1.0, '혼합': 0.976}


최종,said,partial,missing,contradicted
정답,,,,
said,607,1,0,0
partial,1,54,1,0
missing,1,3,71,0
contradicted,3,1,0,22


Key Point 판정 (전체 720 개)


...

슬라이드 Content Coverage 평균 절대 오차 (0~1): {'LLM 1차': 0.043, '최종': 0.014}


,정답 coverage,최종 coverage,최종 오차
scenario,,,
누락,0.596,0.612,0.016
더듬기,0.953,0.952,0.000
발음,0.928,0.935,0.006
실수,0.771,0.793,0.023
요약,0.575,0.565,0.020
의역,0.960,0.960,0.000
인식오류,0.929,0.929,0.000
충실,1.000,1.000,0.000
혼합,0.744,0.802,0.059


틀린 판정 목록 (최종 기준)


,take_id,slide,kp,정답,LLM 1차,최종,검증
89,가상대본1_take3,3,KP3,missing,partial,partial,False
189,가상대본1_take5,8,KP2,contradicted,contradicted,covered,True
284,가상대본1_take8,3,KP3,missing,partial,partial,False
445,가상대본2_take3,4,KP1,partial,covered,covered,True
489,가상대본2_take4,5,KP1,contradicted,contradicted,covered,True
542,가상대본2_take5,8,KP1,contradicted,contradicted,covered,True
640,가상대본2_take8,1,KP3,partial,missing,missing,False
647,가상대본2_take8,3,KP2,covered,partial,partial,False
711,가상대본2_take9,9,KP4,contradicted,contradicted,partial,True


,take_id,slide,fact,STT,정답,최종


## 12. 반복 채점 일관성

같은 STT 를 `STT_CONSISTENCY_SAMPLES` 번(기본 3) 채점해, LLM 판정이 채점할 때마다 얼마나 달라지는지 봅니다.
0번은 10장의 채점이고, 1번부터는 LLM 의미 평가와 교차 검증을 다시 부릅니다 (응답은 캐시해 다시 실행하면 호출하지 않습니다).

| 지표 | 뜻 |
|---|---|
| 대본 문장 판정 일치율 | 모든 채점에서 같은 판정을 받은 문장 비율 (최종) |
| Key Point 판정 일치율 | 모든 채점에서 같은 판정을 받은 Key Point 비율 (LLM 1차 / 최종) |
| 점수 차이 | 같은 슬라이드·발표의 점수가 채점마다 달라진 폭 (최대 − 최소, 100점 만점) |
| 채점별 정확도 | 채점마다 정답 라벨과 비교한 정확도 |

In [19]:
# 같은 STT 를 몇 번 채점해 비교할지. 0번은 10장에서 저장한 채점이고, 1번부터는 LLM 을 다시 부른다 (응답은 캐시해 다시 실행하면 호출하지 않음)
N_EVAL_SAMPLES = int(os.getenv("STT_CONSISTENCY_SAMPLES", "3"))


def spread(values: list[float]) -> float:
    return max(values) - min(values) if values else 0.0


if N_EVAL_SAMPLES < 2:
    print("N_EVAL_SAMPLES 가 1 이라 반복 채점을 건너뜁니다.")
else:
    runs = {0: all_evaluations}
    extra_calls = Counter()
    for k in range(1, N_EVAL_SAMPLES):
        runs[k] = {}
        for take in takes:
            evaluations, stats = evaluate_take(take, eval_llm, verifier_llm, sample=k)
            if stats["failed"]:
                raise RuntimeError(f"반복 채점 {k} / {take.take_id} 실패: {next(iter(stats['failed'].values()))}")
            runs[k][take.take_id] = evaluations
            extra_calls["semantic"] += stats["semantic_calls"]
            extra_calls["verifier"] += stats["verifier_calls"]
    print(f"반복 채점 {N_EVAL_SAMPLES - 1}번 추가 — 이번 실행의 API 호출: {dict(extra_calls)}")

    # 같은 Key Point · 수치 · 슬라이드의 결과를 채점별로 모은다
    kp_status, first_status, sentence_status, slide_cov, take_cov, take_fact = (defaultdict(list) for _ in range(6))
    for k, by_take in runs.items():
        for take_id, evaluations in by_take.items():
            for e in evaluations:
                for s in e.sentences:
                    sentence_status[take_id, e.slide_number, s.sentence_index].append(s.status)
                for r in e.key_points:
                    kp_status[take_id, e.slide_number, r.key_point_id].append(r.status)
                    first_status[take_id, e.slide_number, r.key_point_id].append(r.first_status)
                slide_cov[take_id, e.slide_number].append(100 * e.scores["content_coverage"])
            scores = take_scores(evaluations)
            take_cov[take_id].append(100 * scores["content_coverage"])
            take_fact[take_id].append(100 * (scores["critical_fact_accuracy"] or 0))

    def agree(groups: dict) -> float:
        full = [v for v in groups.values() if len(v) == N_EVAL_SAMPLES]
        return round(sum(len(set(v)) == 1 for v in full) / len(full), 3) if full else float("nan")

    slide_spreads = sorted(spread(v) for v in slide_cov.values())
    print(f"같은 STT 를 {N_EVAL_SAMPLES}번 채점했을 때")
    display(pd.DataFrame([{
        "대본 문장 판정 일치율 (최종)": agree(sentence_status),
        "Key Point 판정 일치율 (LLM 1차)": agree(first_status),
        "Key Point 판정 일치율 (최종)": agree(kp_status),
        "슬라이드 내용 점수 차이 평균": round(sum(slide_spreads) / len(slide_spreads), 2),
        "슬라이드 내용 점수 차이 최대": round(slide_spreads[-1], 2),
        "발표 전체 내용 점수 차이 최대": round(max(spread(v) for v in take_cov.values()), 2),
        "발표 전체 수치 점수 차이 최대": round(max(spread(v) for v in take_fact.values()), 2),
    }]).T.rename(columns={0: "값"}))

    # 채점마다 정답 라벨과 비교
    rows = []
    for k, by_take in runs.items():
        kp_k, fact_k, _, sent_k = build_tables(by_take)
        rows.append({"채점": k, "문장 최종": kp_accuracy(sent_k, "최종")["정확도"],
                     **{f"KP {col}": kp_accuracy(kp_k, col)["정확도"] for col in ("LLM 1차", "최종")},
                     "KP 심각한 오판": kp_accuracy(kp_k, "최종")["심각한 오판 비율"],
                     "사실 정확도 (보류 제외)": fact_accuracy(fact_k), **similar_accuracy(by_take)})
    print("채점별 정확도 (정답 라벨 대비)")
    display(pd.DataFrame(rows).set_index("채점"))

    unstable = [(key, v) for key, v in kp_status.items() if len(set(v)) > 1]
    print(f"채점마다 최종 판정이 달라진 Key Point {len(unstable)}개")
    display(pd.DataFrame([{"take_id": t, "slide": s, "kp": kp, "판정": " / ".join(v)} for (t, s, kp), v in unstable]))

반복 채점 2번 추가 — 이번 실행의 API 호출: {'semantic': 0, 'verifier': 0}
같은 STT 를 3번 채점했을 때


,값
대본 문장 판정 일치율 (최종),0.988
Key Point 판정 일치율 (LLM 1차),0.988
Key Point 판정 일치율 (최종),0.990
슬라이드 내용 점수 차이 평균,0.840
슬라이드 내용 점수 차이 최대,60.000
발표 전체 내용 점수 차이 최대,3.800
발표 전체 수치 점수 차이 최대,0.000


채점별 정확도 (정답 라벨 대비)


,문장 최종,KP LLM 1차,KP 최종,KP 심각한 오판,사실 정확도 (보류 제외),인식 오류 보류율,보류 중 실제 인식 오류
채점,,,,,,,
0,0.986,0.967,0.988,0.004,1.0,1.0,0.897
1,0.990,0.972,0.990,0.006,1.0,1.0,0.897
2,0.992,0.971,0.992,0.004,1.0,1.0,0.897


채점마다 최종 판정이 달라진 Key Point 7개


,take_id,slide,kp,판정
0,가상대본1_take3,3,KP3,partial / missing / missing
1,가상대본1_take8,3,KP3,partial / missing / missing
2,가상대본1_take8,7,KP1,missing / partial / missing
3,가상대본2_take3,4,KP1,covered / partial / partial
4,가상대본2_take7,8,KP2,covered / contradicted / covered
5,가상대본2_take8,7,KP2,missing / missing / partial
6,가상대본2_take9,9,KP4,partial / contradicted / contradicted
